# ERMDSF — Explainable Remaining Useful Life & Maintenance Decision-Support Framework

## Reproducibility and Prototype Note

This notebook reproduces the **ERMDSF research prototype** developed during the thesis.

**Important:** The ERMDSF prototype was developed during an earlier stage of the study. Some saved model artefacts, dashboard values, case-study data, SHAP agreement statistics, and uncertainty outputs therefore correspond to the **earlier 24-feature analysis** rather than the final revised 16-feature experimental pipeline.

These prototype artefacts are retained intentionally to reproduce the ERMDSF interface and functionality presented in the thesis. They should **not** be interpreted as the final numerical results of the revised experiments. The authoritative final 16-feature experimental results and revision analyses are reported in the main-analysis and revision-experiments notebooks.

The interface is a **research prototype**. It is neither certified nor validated for operational maintenance or airworthiness decisions, and it is not a production-ready maintenance system.


In [ ]:
# ============================================================
# ERMDSF REPRODUCIBLE SETUP — GITHUB
# ============================================================
# The notebook automatically retrieves the thesis repository in Google Colab.
# No manual upload of ERMDSF supporting artefacts is required.

import os
import subprocess

REPO_URL = "https://github.com/dilawar1791/turbofan-rul-shap-thesis.git"
REPO_DIR = "/content/turbofan-rul-shap-thesis"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

PATH = os.path.join(REPO_DIR, "ermdsf_artifacts")
dashboard_path = PATH

required_files = [
    "fleet_maintenance_prioritization.csv",
    "operational_case_studies.csv",
    "X_full.csv",
    "train_processed.csv",
    "xgb_rul_model.pkl",
    "xgb_classifier.pkl",
    "dashboard_config.pkl",
]

missing_files = [
    filename for filename in required_files
    if not os.path.exists(os.path.join(PATH, filename))
]

if missing_files:
    raise FileNotFoundError(
        "Missing required ERMDSF artefacts:\n"
        + "\n".join(f" - {filename}" for filename in missing_files)
    )

print("✓ All required ERMDSF artefacts found.")
print("Artefact directory:", PATH)


In [18]:
import pandas as pd
import joblib
# Load CSV files
fleet_priority = pd.read_csv(
    f"{PATH}/fleet_maintenance_prioritization.csv"
)
case_studies = pd.read_csv(
    f"{PATH}/operational_case_studies.csv"
)
X_full = pd.read_csv(
    f"{PATH}/X_full.csv"
)
train = pd.read_csv(
    f"{PATH}/train_processed.csv"
)
# Load trained models and configuration
xgb_cp = joblib.load(
    f"{PATH}/xgb_rul_model.pkl"
)
xgb_clf = joblib.load(
    f"{PATH}/xgb_classifier.pkl"
)
config = joblib.load(
    f"{PATH}/dashboard_config.pkl"
)
conformal_quantile = config["conformal_quantile"]
print("✅ Everything loaded successfully.")
print("fleet_priority:", fleet_priority.shape)
print("case_studies:", case_studies.shape)
print("X_full:", X_full.shape)
print("train:", train.shape)

✅ Everything loaded successfully.
fleet_priority: (20, 12)
case_studies: (3, 13)
X_full: (20631, 25)
train: (20631, 30)


In [3]:
# ============================================================
# CELL 1 — ERMDSF IMPORTS AND ENVIRONMENT CHECK
# ============================================================
try:
    demo.close()
except Exception:
    pass
from datetime import datetime
from pathlib import Path
import os
import warnings
import gradio as gr
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import shap
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)
required_variables = [
    "fleet_priority",
    "case_studies",
    "X_full",
    "train",
    "xgb_cp",
    "xgb_clf",
    "config"
]
missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]
print("=" * 65)
print("ERMDSF DASHBOARD — ENVIRONMENT CHECK")
print("=" * 65)
print(f"Gradio version : {gr.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"SHAP version   : {shap.__version__}")
if missing_variables:
    print("\n❌ Missing required variables:")
    for variable in missing_variables:
        print(f"   - {variable}")
else:
    print("\n✅ All required dashboard variables are available.")
    print(f"Fleet rows      : {len(fleet_priority)}")
    print(f"Feature rows    : {len(X_full)}")
    print(f"Training rows   : {len(train)}")
print("=" * 65)

ERMDSF DASHBOARD — ENVIRONMENT CHECK
Gradio version : 6.27.0
Pandas version : 2.2.3
SHAP version   : 0.52.0

✅ All required dashboard variables are available.
Fleet rows      : 20
Feature rows    : 20631
Training rows   : 20631


In [4]:
# ============================================================
# CELL 2 — DATA PREPARATION AND GLOBAL FLEET METRICS
# ============================================================
# ------------------------------------------------------------
# CLEAN MAIN DATAFRAMES
# ------------------------------------------------------------
fleet_dashboard = fleet_priority.copy()
case_dashboard = case_studies.copy()
X_dashboard = X_full.copy()
train_dashboard = train.copy()
# Remove saved CSV index columns if present
for dataframe in [
    fleet_dashboard,
    case_dashboard,
    X_dashboard,
    train_dashboard
]:
    dataframe.drop(
        columns=["Unnamed: 0"],
        errors="ignore",
        inplace=True
    )
# ------------------------------------------------------------
# STANDARDIZE DATA TYPES
# ------------------------------------------------------------
fleet_dashboard["engine_id"] = (
    fleet_dashboard["engine_id"]
    .astype(int)
)
fleet_dashboard["cycle"] = (
    fleet_dashboard["cycle"]
    .astype(int)
)
fleet_dashboard["maintenance_priority"] = (
    fleet_dashboard["maintenance_priority"]
    .astype(int)
)
case_dashboard["engine_id"] = (
    case_dashboard["engine_id"]
    .astype(int)
)
case_dashboard["cycle"] = (
    case_dashboard["cycle"]
    .astype(int)
)
train_dashboard["engine_id"] = (
    train_dashboard["engine_id"]
    .astype(int)
)
train_dashboard["cycle"] = (
    train_dashboard["cycle"]
    .astype(int)
)
# ------------------------------------------------------------
# SORT FLEET BY OPERATIONAL PRIORITY
# ------------------------------------------------------------
fleet_dashboard = (
    fleet_dashboard
    .sort_values(
        "maintenance_priority"
    )
    .reset_index(drop=True)
)
# ------------------------------------------------------------
# GLOBAL FLEET METRICS
# ------------------------------------------------------------
fleet_size = len(
    fleet_dashboard
)
average_predicted_rul = float(
    fleet_dashboard["predicted_RUL"]
    .mean()
)
median_predicted_rul = float(
    fleet_dashboard["predicted_RUL"]
    .median()
)
average_actual_rul = float(
    fleet_dashboard["actual_RUL"]
    .mean()
)
average_failure_probability = float(
    fleet_dashboard["failure_probability"]
    .mean()
)
average_interval_width = float(
    (
        fleet_dashboard["upper_95"]
        -
        fleet_dashboard["lower_95"]
    )
    .mean()
)
# ------------------------------------------------------------
# RISK COUNTS
# ------------------------------------------------------------
risk_order = [
    "Critical",
    "High",
    "Medium",
    "Low"
]
risk_counts = (
    fleet_dashboard["risk_level"]
    .value_counts()
    .reindex(
        risk_order,
        fill_value=0
    )
)
critical_count = int(
    risk_counts["Critical"]
)
high_count = int(
    risk_counts["High"]
)
medium_count = int(
    risk_counts["Medium"]
)
low_count = int(
    risk_counts["Low"]
)
high_critical_count = (
    critical_count
    +
    high_count
)
# ------------------------------------------------------------
# THRESHOLD-BASED COUNTS
# ------------------------------------------------------------
below_10_count = int(
    (
        fleet_dashboard["predicted_RUL"]
        <= 10
    )
    .sum()
)
below_30_count = int(
    (
        fleet_dashboard["predicted_RUL"]
        <= 30
    )
    .sum()
)
between_30_70_count = int(
    (
        (
            fleet_dashboard["predicted_RUL"]
            > 30
        )
        &
        (
            fleet_dashboard["predicted_RUL"]
            <= 70
        )
    )
    .sum()
)
above_70_count = int(
    (
        fleet_dashboard["predicted_RUL"]
        > 70
    )
    .sum()
)
# ------------------------------------------------------------
# PRIORITY ENGINE DETAILS
# ------------------------------------------------------------
top_priority_row = (
    fleet_dashboard
    .iloc[0]
)
top_priority_engine = int(
    top_priority_row["engine_id"]
)
top_priority_cycle = int(
    top_priority_row["cycle"]
)
top_priority_rul = float(
    top_priority_row["predicted_RUL"]
)
top_priority_probability = float(
    top_priority_row[
        "failure_probability"
    ]
)
top_priority_risk = str(
    top_priority_row["risk_level"]
)
top_priority_recommendation = str(
    top_priority_row["recommendation"]
)
# ------------------------------------------------------------
# TOP FIVE PRIORITY TABLE
# ------------------------------------------------------------
top_five = fleet_dashboard[
    [
        "maintenance_priority",
        "engine_id",
        "cycle",
        "predicted_RUL",
        "failure_probability",
        "risk_level"
    ]
].head(5).copy()
top_five.columns = [
    "Priority",
    "Engine",
    "Cycle",
    "Predicted RUL",
    "Failure Probability",
    "Risk"
]
top_five["Predicted RUL"] = (
    top_five["Predicted RUL"]
    .round(1)
)
top_five["Failure Probability"] = (
    top_five["Failure Probability"]
    .map(
        lambda value:
        f"{value:.1%}"
    )
)
# ------------------------------------------------------------
# GLOBAL CONFIGURATION VALUES
# ------------------------------------------------------------
conformal_quantile = float(
    config.get(
        "conformal_quantile",
        np.nan
    )
)
maintenance_threshold = int(
    config.get(
        "maintenance_threshold",
        30
    )
)
warning_threshold = int(
    config.get(
        "warning_threshold",
        70
    )
)
critical_threshold = int(
    config.get(
        "critical_threshold",
        10
    )
)
cross_model_spearman = float(
    config.get(
        "shap_spearman_models",
        0.981
    )
)
cross_model_jaccard = float(
    config.get(
        "shap_jaccard_models",
        0.778
    )
)
noise_spearman = float(
    config.get(
        "shap_noise_spearman",
        0.996
    )
)
noise_jaccard = float(
    config.get(
        "shap_noise_jaccard",
        1.000
    )
)
# ------------------------------------------------------------
# TIMESTAMP
# ------------------------------------------------------------
last_refresh = (
    datetime.now()
    .strftime(
        "%d %b %Y · %H:%M"
    )
)
# ------------------------------------------------------------
# VALIDATION CHECKS
# ------------------------------------------------------------
required_fleet_columns = [
    "maintenance_priority",
    "engine_id",
    "cycle",
    "actual_RUL",
    "predicted_RUL",
    "lower_95",
    "upper_95",
    "failure_probability",
    "risk_level",
    "RUL_reducing_drivers",
    "recommendation"
]
missing_fleet_columns = [
    column
    for column in required_fleet_columns
    if column not in fleet_dashboard.columns
]
if missing_fleet_columns:
    raise ValueError(
        "Missing fleet columns: "
        + str(missing_fleet_columns)
    )
# ------------------------------------------------------------
# SUMMARY REPORT
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF — GLOBAL FLEET METRICS")
print("=" * 65)
print(
    f"Fleet size                    : "
    f"{fleet_size}"
)
print(
    f"Average predicted RUL         : "
    f"{average_predicted_rul:.2f} cycles"
)
print(
    f"Median predicted RUL          : "
    f"{median_predicted_rul:.2f} cycles"
)
print(
    f"Average failure probability   : "
    f"{average_failure_probability:.2%}"
)
print(
    f"Average interval width        : "
    f"{average_interval_width:.2f} cycles"
)
print()
print(
    f"Critical engines              : "
    f"{critical_count}"
)
print(
    f"High-risk engines             : "
    f"{high_count}"
)
print(
    f"Medium-risk engines           : "
    f"{medium_count}"
)
print(
    f"Low-risk engines              : "
    f"{low_count}"
)
print()
print(
    f"Predicted RUL ≤ 10            : "
    f"{below_10_count}"
)
print(
    f"Predicted RUL ≤ 30            : "
    f"{below_30_count}"
)
print(
    f"Predicted RUL 30–70           : "
    f"{between_30_70_count}"
)
print(
    f"Predicted RUL > 70            : "
    f"{above_70_count}"
)
print()
print(
    f"Top-priority engine           : "
    f"Engine {top_priority_engine}"
)
print(
    f"Top-priority predicted RUL    : "
    f"{top_priority_rul:.2f} cycles"
)
print(
    f"Top-priority failure prob.    : "
    f"{top_priority_probability:.2%}"
)
print(
    f"Top-priority risk             : "
    f"{top_priority_risk}"
)
print("=" * 65)
print("✅ Data preparation completed successfully.")

ERMDSF — GLOBAL FLEET METRICS
Fleet size                    : 20
Average predicted RUL         : 49.16 cycles
Median predicted RUL          : 43.59 cycles
Average failure probability   : 15.68%
Average interval width        : 77.92 cycles

Critical engines              : 0
High-risk engines             : 16
Medium-risk engines           : 0
Low-risk engines              : 4

Predicted RUL ≤ 10            : 0
Predicted RUL ≤ 30            : 1
Predicted RUL 30–70           : 15
Predicted RUL > 70            : 4

Top-priority engine           : Engine 23
Top-priority predicted RUL    : 32.18 cycles
Top-priority failure prob.    : 46.98%
Top-priority risk             : High
✅ Data preparation completed successfully.


In [5]:
# ============================================================
# CELL 3 — GLOBAL CSS AND DESIGN SYSTEM
# ============================================================
custom_css = """
@import url(
'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap'
);
/* ========================================================
   GLOBAL APPLICATION
   ======================================================== */
.gradio-container {
    max-width: 1800px !important;
    margin: auto !important;
    background: #F3F6F9 !important;
    font-family: 'Inter', sans-serif !important;
}
footer {
    display: none !important;
}
body {
    background: #F3F6F9 !important;
}
/* ========================================================
   APPLICATION HEADER
   ======================================================== */
.app-header {
    background:
        linear-gradient(
            115deg,
            #082F49 0%,
            #0D4F78 55%,
            #1769AA 100%
        );
    color: white;
    border-radius: 18px;
    padding: 18px 24px;
    margin-bottom: 18px;
    box-shadow:
        0 10px 30px
        rgba(8, 47, 73, 0.20);
}
.brand-row {
    display: flex;
    align-items: center;
    gap: 14px;
}
.brand-mark {
    width: 48px;
    height: 48px;
    border-radius: 13px;
    background:
        rgba(255,255,255,0.14);
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 25px;
    border:
        1px solid
        rgba(255,255,255,0.22);
}
.brand-title {
    color: white !important;
    font-size: 28px;
    line-height: 1.1;
    font-weight: 800;
    margin: 0;
}
.brand-subtitle {
    color: #DDEFFA !important;
    margin-top: 4px;
    font-size: 12px;
}
.header-status {
    margin-left: auto;
    padding: 8px 13px;
    border-radius: 30px;
    background:
        rgba(46,125,50,0.25);
    border:
        1px solid
        rgba(193,230,194,0.45);
    color: #E9FFE9;
    font-size: 11px;
    font-weight: 700;
}
/* ========================================================
   SIDEBAR
   ======================================================== */
.sidebar-shell {
    background: #082F49;
    border-radius: 17px;
    padding: 20px 14px;
    min-height: 780px;
    box-shadow:
        0 8px 24px
        rgba(8,47,73,0.16);
}
.sidebar-brand {
    color: white;
    padding:
        4px 10px
        18px 10px;
    border-bottom:
        1px solid
        rgba(255,255,255,0.14);
    margin-bottom: 14px;
}
.sidebar-brand-title {
    color: white !important;
    font-size: 18px;
    font-weight: 800;
}
.sidebar-brand-subtitle {
    color: #AFCBDD;
    font-size: 10px;
    margin-top: 3px;
}
/* ========================================================
   NAVIGATION BUTTONS
   ======================================================== */
.nav-btn {
    margin: 4px 0 !important;
}
.nav-btn button {
    width: 100% !important;
    justify-content: flex-start !important;
    background:
        transparent !important;
    color:
        #D9E8F2 !important;
    border:
        1px solid
        transparent !important;
    box-shadow:
        none !important;
    border-radius:
        9px !important;
    padding:
        11px 13px !important;
    font-weight:
        600 !important;
    font-size:
        13px !important;
    text-align:
        left !important;
    transition:
        all 0.18s ease !important;
}
.nav-btn button:hover {
    background:
        rgba(255,255,255,0.10) !important;
    border-color:
        rgba(255,255,255,0.12) !important;
    color:
        white !important;
    transform:
        translateX(3px);
}
.nav-active button {
    background:
        #1769AA !important;
    border-color:
        #3884BC !important;
    color:
        white !important;
    box-shadow:
        0 5px 14px
        rgba(23,105,170,0.35)
        !important;
}
/* ========================================================
   MAIN CONTENT
   ======================================================== */
.main-shell {
    padding:
        4px 8px
        25px 15px;
}
.page-eyebrow {
    color: #1769AA;
    font-size: 11px;
    font-weight: 800;
    letter-spacing: 1.3px;
    margin-bottom: 5px;
}
.page-title {
    color: #123A5A;
    font-size: 28px;
    font-weight: 800;
    margin: 0;
}
.page-description {
    color: #617383;
    font-size: 13px;
    margin-top: 6px;
    margin-bottom: 17px;
}
.refresh-badge {
    color: #70818E;
    font-size: 11px;
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 20px;
    padding:
        8px 12px;
    white-space: nowrap;
    box-shadow:
        0 3px 9px
        rgba(15,61,89,0.04);
}
/* ========================================================
   KPI CARDS
   ======================================================== */
.kpi-grid {
    display: grid;
    grid-template-columns:
        repeat(
            4,
            minmax(0, 1fr)
        );
    gap: 14px;
    margin-bottom: 17px;
}
.kpi-card {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 15px;
    padding: 17px;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
    position: relative;
    overflow: hidden;
}
.kpi-accent {
    position: absolute;
    top: 0;
    left: 0;
    width: 5px;
    height: 100%;
    background: #1769AA;
}
.kpi-label {
    color: #687987;
    font-size: 11px;
    font-weight: 700;
    margin-bottom: 8px;
}
.kpi-value {
    color: #0B3F64;
    font-size: 25px;
    font-weight: 800;
}
.kpi-note {
    color: #8795A0;
    font-size: 10px;
    margin-top: 6px;
}
.kpi-icon {
    width: 42px;
    height: 42px;
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    background: #EAF3FA;
    color: #1769AA;
    font-size: 21px;
    font-weight: 800;
}
/* ========================================================
   PANELS
   ======================================================== */
.panel-card {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 15px;
    padding: 17px;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
.panel-title {
    color: #123A5A;
    font-size: 16px;
    font-weight: 800;
    margin-bottom: 12px;
}
/* ========================================================
   ALERTS
   ======================================================== */
.alert-item {
    display: flex;
    align-items: flex-start;
    gap: 10px;
    padding: 11px;
    border-radius: 11px;
    margin-bottom: 9px;
    font-size: 12px;
    color: #314656;
}
.alert-icon {
    min-width: 25px;
    height: 25px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    color: white;
}
.alert-red {
    background: #FFF0F0;
    border-left: 4px solid #C62828;
}
.alert-red .alert-icon {
    background: #C62828;
}
.alert-orange {
    background: #FFF5E9;
    border-left: 4px solid #EF6C00;
}
.alert-orange .alert-icon {
    background: #EF6C00;
}
.alert-blue {
    background: #EDF6FC;
    border-left: 4px solid #1769AA;
}
.alert-blue .alert-icon {
    background: #1769AA;
}
/* ========================================================
   RECOMMENDATIONS
   ======================================================== */
.recommendation-grid {
    display: grid;
    grid-template-columns:
        repeat(
            3,
            minmax(0, 1fr)
        );
    gap: 12px;
}
.recommendation-card {
    display: flex;
    gap: 12px;
    padding: 13px;
    border-radius: 12px;
    background: #F8FAFC;
    border:
        1px solid #E0E7ED;
}
.recommendation-number {
    min-width: 33px;
    height: 33px;
    border-radius: 10px;
    background: #1769AA;
    color: white;
    display: flex;
    justify-content: center;
    align-items: center;
    font-size: 10px;
    font-weight: 800;
}
.recommendation-card b {
    color: #123A5A;
    font-size: 12px;
}
.recommendation-card p {
    color: #71818E;
    font-size: 10px;
    margin: 5px 0 0 0;
}
/* ========================================================
   ENGINE ANALYSIS
   ======================================================== */
.engine-control-card {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 15px;
    padding: 16px;
    margin-bottom: 15px;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
.engine-summary-grid {
    display: grid;
    grid-template-columns:
        repeat(
            4,
            minmax(0, 1fr)
        );
    gap: 13px;
    margin-bottom: 15px;
}
.engine-metric-card {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 14px;
    padding: 15px;
    position: relative;
    overflow: hidden;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
.engine-metric-accent {
    position: absolute;
    top: 0;
    left: 0;
    width: 5px;
    height: 100%;
    background: #1769AA;
}
.engine-metric-label {
    color: #6A7C8A;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 0.6px;
    margin-bottom: 7px;
}
.engine-metric-value {
    color: #103E60;
    font-size: 23px;
    font-weight: 800;
}
.engine-metric-note {
    color: #8795A0;
    font-size: 10px;
    margin-top: 5px;
}
.risk-badge {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    padding:
        8px 14px;
    border-radius: 30px;
    color: white;
    font-size: 13px;
    font-weight: 800;
}
.risk-low {
    background: #2E7D32;
}
.risk-medium {
    background: #F9A825;
}
.risk-high {
    background: #EF6C00;
}
.risk-critical {
    background: #C62828;
}
.engine-recommendation {
    background:
        linear-gradient(
            110deg,
            #F2F8FC,
            #EAF3FA
        );
    border:
        1px solid #CFE0EC;
    border-left:
        5px solid #1769AA;
    border-radius: 13px;
    padding: 16px;
    margin-bottom: 15px;
}
.engine-recommendation-title {
    color: #123A5A;
    font-size: 15px;
    font-weight: 800;
    margin-bottom: 6px;
}
.engine-recommendation-text {
    color: #405665;
    font-size: 13px;
    line-height: 1.55;
}
.engine-explanation {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 14px;
    padding: 16px;
    margin-top: 14px;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
.engine-explanation h3 {
    color: #123A5A !important;
    margin-top: 0;
    font-size: 16px;
}
.engine-explanation p {
    color: #536875;
    font-size: 12px;
    line-height: 1.6;
    margin-bottom: 0;
}
.engine-plot-card {
    background: white !important;
    border:
        1px solid #DCE5EC !important;
    border-radius:
        15px !important;
    padding:
        7px !important;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
.loading-note {
    background: #EDF6FC;
    border-left:
        4px solid #1769AA;
    color: #405665;
    padding:
        11px 14px;
    border-radius: 10px;
    font-size: 11px;
    margin-bottom: 14px;
}
/* ========================================================
   PLOTLY CHART CONTAINERS
   ======================================================== */
#fleet-risk-chart,
#fleet-rul-chart {
    background: white !important;
    border:
        1px solid
        #DCE5EC !important;
    border-radius:
        15px !important;
    min-height:
        410px !important;
    height:
        410px !important;
    overflow:
        visible !important;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
#fleet-risk-chart > div,
#fleet-rul-chart > div {
    min-height:
        400px !important;
    height:
        400px !important;
}
#fleet-risk-chart .plot-container,
#fleet-rul-chart .plot-container {
    min-height:
        390px !important;
    height:
        390px !important;
}
#fleet-risk-chart .js-plotly-plot,
#fleet-rul-chart .js-plotly-plot {
    width:
        100% !important;
    min-height:
        380px !important;
    height:
        380px !important;
}
#engine-health-gauge {
    min-height:
        330px !important;
    height:
        330px !important;
}
#engine-shap-bar {
    min-height:
        410px !important;
    height:
        410px !important;
}
#engine-waterfall {
    min-height:
        460px !important;
}
/* ========================================================
   TABLES
   ======================================================== */
.table-wrap {
    background: white;
    border:
        1px solid #DCE5EC;
    border-radius: 14px;
    padding: 8px;
    box-shadow:
        0 5px 15px
        rgba(15,61,89,0.06);
}
/* ========================================================
   INFO / WARNING / SUCCESS BOXES
   ======================================================== */
.info-box {
    background: #EDF6FC;
    border-left:
        5px solid #1769AA;
    border-radius: 11px;
    padding: 13px;
    color: #34566E;
    font-size: 11px;
    line-height: 1.6;
}
.warning-box {
    background: #FFF8E7;
    border-left:
        5px solid #F9A825;
    border-radius: 11px;
    padding: 13px;
    color: #625523;
    font-size: 11px;
    line-height: 1.6;
}
.danger-box {
    background: #FFF0F0;
    border-left:
        5px solid #C62828;
    border-radius: 11px;
    padding: 13px;
    color: #6D2929;
    font-size: 11px;
    line-height: 1.6;
}
.success-box {
    background: #EDF8EE;
    border-left:
        5px solid #2E7D32;
    border-radius: 11px;
    padding: 13px;
    color: #285C2B;
    font-size: 11px;
    line-height: 1.6;
}
/* ========================================================
   RESPONSIVE DESIGN
   ======================================================== */
@media (max-width: 1000px) {
    .kpi-grid {
        grid-template-columns:
            repeat(
                2,
                minmax(0, 1fr)
            );
    }
    .engine-summary-grid {
        grid-template-columns:
            repeat(
                2,
                minmax(0, 1fr)
            );
    }
    .recommendation-grid {
        grid-template-columns:
            1fr;
    }
    .header-status {
        display: none;
    }
}
"""
# ------------------------------------------------------------
# DESIGN TOKENS
# ------------------------------------------------------------
COLORS = {
    "navy": "#082F49",
    "blue": "#1769AA",
    "success": "#2E7D32",
    "warning": "#F9A825",
    "high": "#EF6C00",
    "critical": "#C62828",
    "background": "#F3F6F9",
    "card": "#FFFFFF",
    "text": "#123A5A",
    "muted": "#617383"
}
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF GLOBAL DESIGN SYSTEM")
print("=" * 65)
print("✅ Global CSS created successfully.")
print("✅ Home-page styles included.")
print("✅ Engine Analysis styles included.")
print("✅ Planner and common utility styles included.")
print("✅ Responsive styles included.")
print("=" * 65)

ERMDSF GLOBAL DESIGN SYSTEM
✅ Global CSS created successfully.
✅ Home-page styles included.
✅ Engine Analysis styles included.
✅ Planner and common utility styles included.
✅ Responsive styles included.


In [6]:
# ============================================================
# CELL 4 — HOME PAGE FUNCTIONS AND VISUALS
# ============================================================
# ------------------------------------------------------------
# RISK COLOURS
# ------------------------------------------------------------
RISK_COLORS = {
    "Critical": COLORS["critical"],
    "High": COLORS["high"],
    "Medium": COLORS["warning"],
    "Low": COLORS["success"]
}
# ------------------------------------------------------------
# KPI CARD HTML
# ------------------------------------------------------------
def create_home_kpi_html():
    """
    Create the four executive fleet KPI cards.
    """
    return f"""
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:flex-start;
                gap:12px;
            ">
                <div>
                    <div class="kpi-label">
                        MONITORED FLEET
                    </div>
                    <div class="kpi-value">
                        {fleet_size} engines
                    </div>
                    <div class="kpi-note">
                        Current fleet snapshot
                    </div>
                </div>
                <div class="kpi-icon">
                    ✈
                </div>
            </div>
        </div>
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:flex-start;
                gap:12px;
            ">
                <div>
                    <div class="kpi-label">
                        AVERAGE PREDICTED RUL
                    </div>
                    <div class="kpi-value">
                        {average_predicted_rul:.1f}
                        cycles
                    </div>
                    <div class="kpi-note">
                        Median:
                        {median_predicted_rul:.1f}
                        cycles
                    </div>
                </div>
                <div class="kpi-icon">
                    ◷
                </div>
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['high']};
                "
            >
            </div>
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:flex-start;
                gap:12px;
            ">
                <div>
                    <div class="kpi-label">
                        HIGH / CRITICAL
                    </div>
                    <div class="kpi-value">
                        {high_critical_count}
                    </div>
                    <div class="kpi-note">
                        Engines requiring attention
                    </div>
                </div>
                <div
                    class="kpi-icon"
                    style="
                        color:{COLORS['high']};
                        background:#FFF2E7;
                    "
                >
                    !
                </div>
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['critical']};
                "
            >
            </div>
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:flex-start;
                gap:12px;
            ">
                <div>
                    <div class="kpi-label">
                        NEXT PRIORITY
                    </div>
                    <div class="kpi-value">
                        Engine
                        {top_priority_engine}
                    </div>
                    <div class="kpi-note">
                        Predicted RUL:
                        {top_priority_rul:.1f}
                        cycles
                    </div>
                </div>
                <div
                    class="kpi-icon"
                    style="
                        color:{COLORS['critical']};
                        background:#FDECEC;
                    "
                >
                    ⚒
                </div>
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# FLEET RISK DONUT CHART
# ------------------------------------------------------------
def create_fleet_risk_donut():
    """
    Create the fleet risk-distribution donut chart.
    """
    risk_chart_data = pd.DataFrame({
        "Risk Level": risk_order,
        "Engines": [
            critical_count,
            high_count,
            medium_count,
            low_count
        ]
    })
    figure = go.Figure()
    figure.add_trace(
        go.Pie(
            labels=risk_chart_data[
                "Risk Level"
            ],
            values=risk_chart_data[
                "Engines"
            ],
            hole=0.62,
            marker=dict(
                colors=[
                    RISK_COLORS[level]
                    for level
                    in risk_order
                ],
                line=dict(
                    color="white",
                    width=2
                )
            ),
            textinfo="label+value",
            textfont=dict(
                size=12
            ),
            hovertemplate=(
                "<b>%{label}</b><br>"
                "Engines: %{value}<br>"
                "Fleet share: %{percent}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text="Fleet Risk Distribution",
            x=0.04,
            y=0.96,
            font=dict(
                size=19,
                color=COLORS["text"],
                family="Inter"
            )
        ),
        annotations=[
            dict(
                text=(
                    "<span style='font-size:11px'>"
                    "FLEET HEALTH"
                    "</span>"
                    f"<br><b>{fleet_size}</b>"
                    "<br>"
                    "<span style='font-size:10px'>"
                    "MONITORED ENGINES"
                    "</span>"
                ),
                x=0.5,
                y=0.5,
                font=dict(
                    size=17,
                    color=COLORS["text"]
                ),
                align="center",
                showarrow=False
            )
        ],
        showlegend=True,
        legend=dict(
            orientation="h",
            y=-0.04,
            x=0.5,
            xanchor="center",
            font=dict(
                size=11
            )
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=20,
            r=20,
            t=60,
            b=45
        ),
        height=380,
        autosize=True
    )
    return figure
# ------------------------------------------------------------
# RUL DISTRIBUTION CHART
# ------------------------------------------------------------
def create_rul_histogram():
    """
    Create the fleet predicted-RUL distribution histogram.
    """
    figure = px.histogram(
        fleet_dashboard,
        x="predicted_RUL",
        nbins=8,
        labels={
            "predicted_RUL":
            "Predicted RUL (cycles)"
        }
    )
    figure.update_traces(
        marker_color=COLORS["blue"],
        marker_line_color="white",
        marker_line_width=1,
        hovertemplate=(
            "Predicted RUL: %{x:.1f}<br>"
            "Number of engines: %{y}"
            "<extra></extra>"
        )
    )
    figure.add_vline(
        x=critical_threshold,
        line_dash="dash",
        line_color=COLORS["critical"],
        line_width=2,
        annotation_text=(
            f"{critical_threshold}-cycle "
            "critical threshold"
        ),
        annotation_position="top left"
    )
    figure.add_vline(
        x=maintenance_threshold,
        line_dash="dash",
        line_color=COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "maintenance threshold"
        ),
        annotation_position="top right"
    )
    figure.add_vline(
        x=warning_threshold,
        line_dash="dash",
        line_color=COLORS["success"],
        line_width=2,
        annotation_text=(
            f"{warning_threshold}-cycle "
            "monitoring threshold"
        ),
        annotation_position="top right"
    )
    figure.update_layout(
        title=dict(
            text="Predicted RUL Distribution",
            x=0.04,
            y=0.96,
            font=dict(
                size=19,
                color=COLORS["text"],
                family="Inter"
            )
        ),
        xaxis_title=(
            "Predicted Remaining Useful Life "
            "(cycles)"
        ),
        yaxis_title="Number of engines",
        paper_bgcolor="white",
        plot_bgcolor="white",
        bargap=0.08,
        margin=dict(
            l=55,
            r=20,
            t=60,
            b=55
        ),
        height=380,
        autosize=True
    )
    figure.update_xaxes(
        gridcolor="#E8EEF3",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor="#E8EEF3",
        zeroline=False
    )
    return figure
# ------------------------------------------------------------
# LIVE OPERATIONAL ALERTS
# ------------------------------------------------------------
def create_home_alerts_html():
    """
    Generate operational alerts from the current fleet data.
    """
    alerts = []
    alerts.append(
        f"""
        <div class="alert-item alert-orange">
            <div class="alert-icon">
                !
            </div>
            <div>
                <b>
                    Engine {top_priority_engine}
                </b>
                is the current highest maintenance
                priority with a predicted RUL of
                <b>
                    {top_priority_rul:.1f} cycles
                </b>.
            </div>
        </div>
        """
    )
    if below_30_count > 0:
        affected_engines = (
            fleet_dashboard[
                fleet_dashboard[
                    "predicted_RUL"
                ]
                <= maintenance_threshold
            ]["engine_id"]
            .astype(int)
            .astype(str)
            .head(5)
            .tolist()
        )
        engine_text = ", ".join(
            affected_engines
        )
        alerts.append(
            f"""
            <div class="alert-item alert-red">
                <div class="alert-icon">
                    !
                </div>
                <div>
                    <b>
                        {below_30_count}
                        engine(s)
                    </b>
                    have predicted RUL at or below
                    {maintenance_threshold} cycles.
                    First affected engines:
                    <b>
                        {engine_text}
                    </b>.
                </div>
            </div>
            """
        )
    if high_critical_count > 0:
        alerts.append(
            f"""
            <div class="alert-item alert-blue">
                <div class="alert-icon">
                    i
                </div>
                <div>
                    <b>
                        {high_critical_count}
                        of
                        {fleet_size}
                        engines
                    </b>
                    are classified as High or Critical
                    under the experimental operational
                    risk matrix.
                </div>
            </div>
            """
        )
    if above_70_count > 0:
        alerts.append(
            f"""
            <div class="alert-item alert-blue">
                <div class="alert-icon">
                    ✓
                </div>
                <div>
                    <b>
                        {above_70_count}
                        engine(s)
                    </b>
                    have predicted RUL above
                    {warning_threshold} cycles
                    and remain in the routine-monitoring
                    range.
                </div>
            </div>
            """
        )
    return "".join(alerts)
# ------------------------------------------------------------
# RECOMMENDED ACTIONS
# ------------------------------------------------------------
def create_home_recommendations_html():
    """
    Create three operational recommendation cards.
    """
    return f"""
    <div class="recommendation-grid">
        <div class="recommendation-card">
            <div class="recommendation-number">
                01
            </div>
            <div>
                <b>
                    Inspect Engine
                    {top_priority_engine}
                </b>
                <p>
                    Current highest fleet priority
                    with predicted RUL of
                    {top_priority_rul:.1f} cycles.
                </p>
            </div>
        </div>
        <div class="recommendation-card">
            <div class="recommendation-number">
                02
            </div>
            <div>
                <b>
                    Allocate inspection capacity
                </b>
                <p>
                    Review the top five fleet-priority
                    engines in the Maintenance Planner.
                </p>
            </div>
        </div>
        <div class="recommendation-card">
            <div class="recommendation-number">
                03
            </div>
            <div>
                <b>
                    Verify SHAP degradation drivers
                </b>
                <p>
                    Confirm the dominant RUL-reducing
                    sensors before planning action.
                </p>
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# PLACEHOLDER PAGE HELPER
# ------------------------------------------------------------
def placeholder_page_html(
    title,
    description
):
    """
    Temporary reusable placeholder for unfinished pages.
    """
    return f"""
    <div class="page-eyebrow">
        ERMDSF MODULE
    </div>
    <h1 class="page-title">
        {title}
    </h1>
    <p class="page-description">
        {description}
    </p>
    <div class="panel-card">
        <div class="panel-title">
            Module under development
        </div>
        <p style="
            color:#6C7C88;
            margin-bottom:0;
        ">
            This module will be added in a later
            notebook cell before the final dashboard
            is launched.
        </p>
    </div>
    """
# ------------------------------------------------------------
# BUILD HOME PAGE OBJECTS
# ------------------------------------------------------------
home_kpi_html = (
    create_home_kpi_html()
)
fleet_risk_donut = (
    create_fleet_risk_donut()
)
fleet_rul_histogram = (
    create_rul_histogram()
)
home_alerts_html = (
    create_home_alerts_html()
)
home_recommendations_html = (
    create_home_recommendations_html()
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF HOME PAGE COMPONENTS")
print("=" * 65)
print("✅ KPI cards created.")
print("✅ Fleet-risk donut chart created.")
print("✅ RUL histogram created.")
print("✅ Operational alerts created.")
print("✅ Recommended-action cards created.")
print("✅ Placeholder helper created.")
print("=" * 65)

ERMDSF HOME PAGE COMPONENTS
✅ KPI cards created.
✅ Fleet-risk donut chart created.
✅ RUL histogram created.
✅ Operational alerts created.
✅ Recommended-action cards created.
✅ Placeholder helper created.


In [7]:
# ============================================================
# CELL 5 — ENGINE ANALYSIS AND SHAP FUNCTIONS
# ============================================================
# ------------------------------------------------------------
# PREPARE ENGINE FEATURE TABLES
# ------------------------------------------------------------
engine_X = X_dashboard.copy()
engine_train = train_dashboard.copy()
# Ensure both tables use matching row indexes
engine_X = engine_X.reset_index(drop=True)
engine_train = engine_train.reset_index(drop=True)
if len(engine_X) != len(engine_train):
    raise ValueError(
        "X_dashboard and train_dashboard have different "
        "numbers of rows and cannot be mapped safely."
    )
# ------------------------------------------------------------
# IDENTIFY FEATURES EXPECTED BY THE XGBOOST MODEL
# ------------------------------------------------------------
model_feature_names = None
# First preference: sklearn-compatible feature names
if hasattr(xgb_cp, "feature_names_in_"):
    model_feature_names = list(
        xgb_cp.feature_names_in_
    )
# Second preference: native XGBoost feature names
if not model_feature_names:
    try:
        booster_feature_names = (
            xgb_cp
            .get_booster()
            .feature_names
        )
        if booster_feature_names:
            model_feature_names = list(
                booster_feature_names
            )
    except Exception:
        model_feature_names = None
# Final fallback
if not model_feature_names:
    excluded_columns = {
        "engine_id",
        "cycle",
        "RUL",
        "max_cycle",
        "failure_label",
        "label"
    }
    model_feature_names = [
        column
        for column in engine_X.columns
        if column not in excluded_columns
    ]
# ------------------------------------------------------------
# VALIDATE FEATURE AVAILABILITY
# ------------------------------------------------------------
missing_features = [
    feature
    for feature in model_feature_names
    if feature not in engine_X.columns
]
if missing_features:
    raise ValueError(
        "The following model features are missing "
        "from X_dashboard:\n"
        + str(missing_features)
    )
engine_feature_table = engine_X[
    model_feature_names
].copy()
# Convert all model inputs to numeric values
for column in engine_feature_table.columns:
    engine_feature_table[column] = (
        pd.to_numeric(
            engine_feature_table[column],
            errors="coerce"
        )
    )
if engine_feature_table.isna().any().any():
    missing_input_columns = (
        engine_feature_table
        .columns[
            engine_feature_table
            .isna()
            .any()
        ]
        .tolist()
    )
    raise ValueError(
        "Missing or non-numeric values were found "
        "in these model-input columns:\n"
        + str(missing_input_columns)
    )
# ------------------------------------------------------------
# CREATE ENGINE–CYCLE TO ROW-INDEX MAPPING
# ------------------------------------------------------------
engine_cycle_to_index = {}
for row_index, row in engine_train[
    [
        "engine_id",
        "cycle"
    ]
].iterrows():
    key = (
        int(row["engine_id"]),
        int(row["cycle"])
    )
    engine_cycle_to_index[key] = int(
        row_index
    )
def find_engine_row_index(
    engine_id,
    cycle
):
    """
    Return the exact row index associated with
    one engine and cycle.
    """
    key = (
        int(engine_id),
        int(cycle)
    )
    if key not in engine_cycle_to_index:
        raise KeyError(
            f"No row was found for Engine {engine_id}, "
            f"Cycle {cycle}."
        )
    return engine_cycle_to_index[key]
def get_engine_snapshot(
    engine_id
):
    """
    Return the operational fleet row for one engine.
    """
    selected_rows = fleet_dashboard[
        fleet_dashboard["engine_id"].astype(int)
        ==
        int(engine_id)
    ]
    if selected_rows.empty:
        raise KeyError(
            f"Engine {engine_id} does not exist "
            "in the fleet-priority table."
        )
    return selected_rows.iloc[0].copy()
def get_engine_input_row(
    engine_id,
    cycle
):
    """
    Return the exact model-input row for one engine snapshot.
    """
    row_index = find_engine_row_index(
        engine_id,
        cycle
    )
    X_selected = (
        engine_feature_table
        .iloc[
            [
                row_index
            ]
        ]
        .copy()
    )
    return row_index, X_selected
# ------------------------------------------------------------
# CREATE SHAP EXPLAINER
# ------------------------------------------------------------
engine_shap_explainer = shap.TreeExplainer(
    xgb_cp
)
def calculate_engine_shap(
    X_selected
):
    """
    Compute local SHAP values and return:
    - ranked contribution table;
    - SHAP Explanation object.
    """
    raw_shap_values = (
        engine_shap_explainer
        .shap_values(
            X_selected
        )
    )
    raw_shap_values = np.asarray(
        raw_shap_values
    )
    # Standard regression shape:
    # rows × features
    if raw_shap_values.ndim == 2:
        local_values = (
            raw_shap_values[0]
        )
    # Handle possible extra output dimension
    elif raw_shap_values.ndim == 3:
        local_values = (
            raw_shap_values[
                0,
                :,
                0
            ]
        )
    else:
        raise ValueError(
            "Unexpected SHAP array shape: "
            f"{raw_shap_values.shape}"
        )
    expected_value = (
        engine_shap_explainer
        .expected_value
    )
    expected_value = float(
        np.asarray(
            expected_value
        )
        .reshape(-1)[0]
    )
    explanation = shap.Explanation(
        values=local_values,
        base_values=expected_value,
        data=(
            X_selected
            .iloc[0]
            .to_numpy()
        ),
        feature_names=list(
            X_selected.columns
        )
    )
    contribution_table = pd.DataFrame({
        "Feature":
            X_selected.columns,
        "SHAP Value":
            local_values,
        "Absolute Contribution":
            np.abs(
                local_values
            )
    })
    contribution_table = (
        contribution_table
        .sort_values(
            "Absolute Contribution",
            ascending=False
        )
        .reset_index(drop=True)
    )
    return (
        contribution_table,
        explanation
    )
# ------------------------------------------------------------
# PRECOMPUTE SHAP VALUES FOR FLEET SNAPSHOTS
# ------------------------------------------------------------
engine_cache = {}
print("=" * 65)
print("PREPARING ENGINE SHAP CACHE")
print("=" * 65)
for cache_position, fleet_row in (
    fleet_dashboard.iterrows()
):
    engine_id = int(
        fleet_row["engine_id"]
    )
    cycle = int(
        fleet_row["cycle"]
    )
    row_index, X_selected = (
        get_engine_input_row(
            engine_id,
            cycle
        )
    )
    contribution_table, explanation = (
        calculate_engine_shap(
            X_selected
        )
    )
    model_prediction = float(
        xgb_cp.predict(
            X_selected
        )[0]
    )
    engine_cache[engine_id] = {
        "fleet_row":
            fleet_row.copy(),
        "row_index":
            row_index,
        "X_selected":
            X_selected.copy(),
        "contribution_table":
            contribution_table,
        "explanation":
            explanation,
        "model_prediction":
            model_prediction
    }
    print(
        f"Cached Engine {engine_id:>3} "
        f"— Cycle {cycle:>3} "
        f"({cache_position + 1}/{fleet_size})"
    )
print("=" * 65)
print(
    f"✅ SHAP cache created for "
    f"{len(engine_cache)} engines."
)
print("=" * 65)
# ------------------------------------------------------------
# ENGINE HEALTH GAUGE
# ------------------------------------------------------------
def create_engine_health_gauge(
    predicted_rul
):
    """
    Create an operational RUL gauge.
    """
    predicted_rul = float(
        predicted_rul
    )
    figure = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=predicted_rul,
            number={
                "suffix": " cycles",
                "font": {
                    "size": 27,
                    "color": COLORS["text"]
                }
            },
            title={
                "text":
                    "Predicted Remaining Useful Life",
                "font": {
                    "size": 16,
                    "color": COLORS["text"]
                }
            },
            gauge={
                "axis": {
                    "range": [0, 125],
                    "tickwidth": 1,
                    "tickcolor":
                        "#7C8C98"
                },
                "bar": {
                    "color":
                        COLORS["blue"],
                    "thickness":
                        0.30
                },
                "bgcolor":
                    "white",
                "borderwidth":
                    0,
                "steps": [
                    {
                        "range": [
                            0,
                            critical_threshold
                        ],
                        "color":
                            "#F6C6C6"
                    },
                    {
                        "range": [
                            critical_threshold,
                            maintenance_threshold
                        ],
                        "color":
                            "#FFD4B2"
                    },
                    {
                        "range": [
                            maintenance_threshold,
                            warning_threshold
                        ],
                        "color":
                            "#FFE9A9"
                    },
                    {
                        "range": [
                            warning_threshold,
                            125
                        ],
                        "color":
                            "#CDE8CF"
                    }
                ],
                "threshold": {
                    "line": {
                        "color":
                            COLORS["critical"],
                        "width":
                            4
                    },
                    "thickness":
                        0.80,
                    "value":
                        predicted_rul
                }
            }
        )
    )
    figure.update_layout(
        paper_bgcolor="white",
        margin=dict(
            l=30,
            r=30,
            t=55,
            b=25
        ),
        height=310
    )
    return figure
# ------------------------------------------------------------
# LOCAL SHAP BAR CHART
# ------------------------------------------------------------
def create_local_shap_bar(
    contribution_table,
    top_n=10
):
    """
    Create a horizontal chart of the most important
    local SHAP contributions.
    """
    chart_data = (
        contribution_table
        .head(top_n)
        .sort_values(
            "SHAP Value",
            ascending=True
        )
        .copy()
    )
    chart_colors = [
        (
            COLORS["critical"]
            if value < 0
            else COLORS["blue"]
        )
        for value
        in chart_data[
            "SHAP Value"
        ]
    ]
    figure = go.Figure(
        go.Bar(
            x=chart_data[
                "SHAP Value"
            ],
            y=chart_data[
                "Feature"
            ],
            orientation="h",
            marker_color=
                chart_colors,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "SHAP contribution: "
                "%{x:+.3f} cycles"
                "<extra></extra>"
            )
        )
    )
    figure.add_vline(
        x=0,
        line_width=1,
        line_color="#7B8B97"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Top Local SHAP "
                "Contributions"
            ),
            x=0.04,
            font=dict(
                size=17,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Contribution to predicted RUL "
            "(cycles)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=90,
            r=25,
            t=55,
            b=50
        ),
        height=390,
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor=(
            "rgba(0,0,0,0)"
        )
    )
    return figure
# ------------------------------------------------------------
# LOCAL SHAP WATERFALL
# ------------------------------------------------------------
def create_local_waterfall(
    explanation
):
    """
    Create a Matplotlib SHAP waterfall plot.
    """
    plt.close("all")
    shap.plots.waterfall(
        explanation,
        max_display=12,
        show=False
    )
    figure = plt.gcf()
    figure.set_size_inches(
        10,
        5.5
    )
    figure.tight_layout()
    return figure
# ------------------------------------------------------------
# ENGINE SUMMARY HTML
# ------------------------------------------------------------
def create_engine_summary_html(
    selected_row
):
    """
    Build the four engine-summary cards and
    conformal uncertainty panel.
    """
    risk = str(
        selected_row[
            "risk_level"
        ]
    )
    risk_class = (
        risk.lower()
    )
    interval_width = (
        float(
            selected_row[
                "upper_95"
            ]
        )
        -
        float(
            selected_row[
                "lower_95"
            ]
        )
    )
    return f"""
    <div class="engine-summary-grid">
        <div class="engine-metric-card">
            <div class="engine-metric-accent">
            </div>
            <div class="engine-metric-label">
                SELECTED ENGINE
            </div>
            <div class="engine-metric-value">
                Engine
                {int(selected_row['engine_id'])}
            </div>
            <div class="engine-metric-note">
                Snapshot cycle:
                {int(selected_row['cycle'])}
            </div>
        </div>
        <div class="engine-metric-card">
            <div class="engine-metric-accent">
            </div>
            <div class="engine-metric-label">
                PREDICTED RUL
            </div>
            <div class="engine-metric-value">
                {selected_row['predicted_RUL']:.1f}
                cycles
            </div>
            <div class="engine-metric-note">
                Actual RUL:
                {selected_row['actual_RUL']:.1f}
                cycles
            </div>
        </div>
        <div class="engine-metric-card">
            <div
                class="engine-metric-accent"
                style="
                    background:
                    {COLORS['high']};
                "
            >
            </div>
            <div class="engine-metric-label">
                FAILURE PROBABILITY
            </div>
            <div class="engine-metric-value">
                {
                    selected_row[
                        'failure_probability'
                    ]
                    :.1%
                }
            </div>
            <div class="engine-metric-note">
                Probability of RUL below
                {maintenance_threshold} cycles
            </div>
        </div>
        <div class="engine-metric-card">
            <div
                class="engine-metric-accent"
                style="
                    background:
                    {RISK_COLORS.get(
                        risk,
                        COLORS['blue']
                    )};
                "
            >
            </div>
            <div class="engine-metric-label">
                OPERATIONAL RISK
            </div>
            <div>
                <span
                    class="
                        risk-badge
                        risk-{risk_class}
                    "
                >
                    {risk.upper()}
                </span>
            </div>
            <div class="engine-metric-note">
                Fleet priority:
                #{
                    int(
                        selected_row[
                            'maintenance_priority'
                        ]
                    )
                }
            </div>
        </div>
    </div>
    <div class="panel-card">
        <div class="panel-title">
            Research Uncertainty Interval
        </div>
        <div style="
            color:{COLORS['text']};
            font-size:22px;
            font-weight:800;
        ">
            {
                selected_row[
                    'lower_95'
                ]
                :.1f
            }
            –
            {
                selected_row[
                    'upper_95'
                ]
                :.1f
            }
            cycles
        </div>
        <div style="
            color:#7B8B97;
            font-size:10px;
            margin-top:5px;
        ">
            Interval width:
            {interval_width:.1f} cycles.
            The conformal interval achieved good
            statistical coverage but was operationally
            wide. It should be used as supporting
            evidence rather than an automatic trigger.
        </div>
    </div>
    """
# ------------------------------------------------------------
# MAINTENANCE RECOMMENDATION HTML
# ------------------------------------------------------------
def create_engine_recommendation_html(
    selected_row
):
    """
    Create the maintenance-recommendation panel.
    """
    return f"""
    <div class="engine-recommendation">
        <div class="engine-recommendation-title">
            Maintenance Recommendation
        </div>
        <div class="engine-recommendation-text">
            {
                selected_row[
                    'recommendation'
                ]
            }
        </div>
    </div>
    """
# ------------------------------------------------------------
# NATURAL-LANGUAGE SHAP INTERPRETATION
# ------------------------------------------------------------
def create_engine_interpretation_html(
    selected_row,
    contribution_table
):
    """
    Convert local SHAP results into concise
    operational interpretation text.
    """
    negative_features = (
        contribution_table[
            contribution_table[
                "SHAP Value"
            ]
            <
            0
        ]
        .head(3)
    )
    positive_features = (
        contribution_table[
            contribution_table[
                "SHAP Value"
            ]
            >
            0
        ]
        .head(3)
    )
    if not negative_features.empty:
        negative_text = ", ".join(
            (
                f"{row['Feature']} "
                f"({row['SHAP Value']:+.2f})"
            )
            for _, row
            in negative_features.iterrows()
        )
    else:
        negative_text = (
            "No strong negative sensor "
            "contribution was identified"
        )
    if not positive_features.empty:
        positive_text = ", ".join(
            (
                f"{row['Feature']} "
                f"({row['SHAP Value']:+.2f})"
            )
            for _, row
            in positive_features.iterrows()
        )
    else:
        positive_text = (
            "No strong positive sensor "
            "contribution was identified"
        )
    prediction_error = (
        float(
            selected_row[
                "predicted_RUL"
            ]
        )
        -
        float(
            selected_row[
                "actual_RUL"
            ]
        )
    )
    if prediction_error > 5:
        prediction_comment = (
            "The model overestimates actual RUL "
            f"by {abs(prediction_error):.1f} cycles."
        )
    elif prediction_error < -5:
        prediction_comment = (
            "The model provides a conservative "
            "estimate and underestimates actual RUL "
            f"by {abs(prediction_error):.1f} cycles."
        )
    else:
        prediction_comment = (
            "The predicted and actual RUL values "
            "are closely aligned for this snapshot."
        )
    return f"""
    <div class="engine-explanation">
        <h3>
            Explainable Maintenance Interpretation
        </h3>
        <p>
            <b>RUL-reducing drivers:</b>
            {negative_text}
            <br><br>
            <b>RUL-increasing drivers:</b>
            {positive_text}
            <br><br>
            <b>Prediction assessment:</b>
            {prediction_comment}
            <br><br>
            Negative SHAP values reduce the
            predicted remaining useful life.
            Positive SHAP values increase it.
        </p>
    </div>
    """
# ------------------------------------------------------------
# ENGINE ANALYSIS CALLBACK
# ------------------------------------------------------------
def update_engine_analysis(
    engine_id
):
    """
    Produce every dynamic output for the selected engine.
    """
    engine_id = int(
        engine_id
    )
    if engine_id not in engine_cache:
        raise gr.Error(
            f"Engine {engine_id} is not available "
            "in the analysis cache."
        )
    cached = engine_cache[
        engine_id
    ]
    selected_row = cached[
        "fleet_row"
    ]
    contribution_table = cached[
        "contribution_table"
    ]
    explanation = cached[
        "explanation"
    ]
    summary_html = (
        create_engine_summary_html(
            selected_row
        )
    )
    gauge = (
        create_engine_health_gauge(
            selected_row[
                "predicted_RUL"
            ]
        )
    )
    recommendation_html = (
        create_engine_recommendation_html(
            selected_row
        )
    )
    display_table = (
        contribution_table[
            [
                "Feature",
                "SHAP Value"
            ]
        ]
        .head(10)
        .copy()
    )
    display_table[
        "SHAP Value"
    ] = (
        display_table[
            "SHAP Value"
        ]
        .round(4)
    )
    shap_bar = (
        create_local_shap_bar(
            contribution_table,
            top_n=10
        )
    )
    waterfall = (
        create_local_waterfall(
            explanation
        )
    )
    interpretation_html = (
        create_engine_interpretation_html(
            selected_row,
            contribution_table
        )
    )
    return (
        summary_html,
        gauge,
        recommendation_html,
        display_table,
        shap_bar,
        waterfall,
        interpretation_html
    )
# ------------------------------------------------------------
# VALIDATE MODEL PREDICTIONS AGAINST SAVED OUTPUTS
# ------------------------------------------------------------
prediction_differences = []
for engine_id, cached in engine_cache.items():
    saved_prediction = float(
        cached[
            "fleet_row"
        ][
            "predicted_RUL"
        ]
    )
    recalculated_prediction = float(
        cached[
            "model_prediction"
        ]
    )
    prediction_differences.append(
        abs(
            saved_prediction
            -
            recalculated_prediction
        )
    )
maximum_prediction_difference = max(
    prediction_differences
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF ENGINE ANALYSIS COMPONENTS")
print("=" * 65)
print(
    f"✅ Model features detected: "
    f"{len(model_feature_names)}"
)
print(
    f"✅ Engine-cycle mappings: "
    f"{len(engine_cycle_to_index)}"
)
print(
    f"✅ Cached fleet engines: "
    f"{len(engine_cache)}"
)
print(
    "✅ Engine gauge function created."
)
print(
    "✅ Local SHAP bar function created."
)
print(
    "✅ SHAP waterfall function created."
)
print(
    "✅ Engine interpretation function created."
)
print(
    f"Maximum saved/recalculated prediction difference: "
    f"{maximum_prediction_difference:.6f}"
)
print("=" * 65)

PREPARING ENGINE SHAP CACHE
Cached Engine  23 — Cycle 135 (1/20)
Cached Engine  84 — Cycle 214 (2/20)
Cached Engine  91 — Cycle 108 (3/20)
Cached Engine  19 — Cycle 127 (4/20)
Cached Engine  71 — Cycle 167 (5/20)
Cached Engine  40 — Cycle 151 (6/20)
Cached Engine  13 — Cycle 131 (7/20)
Cached Engine  32 — Cycle 153 (8/20)
Cached Engine  34 — Cycle 156 (9/20)
Cached Engine  77 — Cycle 124 (10/20)
Cached Engine   5 — Cycle 216 (11/20)
Cached Engine  81 — Cycle 192 (12/20)
Cached Engine  54 — Cycle 206 (13/20)
Cached Engine   1 — Cycle 154 (14/20)
Cached Engine  11 — Cycle 192 (15/20)
Cached Engine  45 — Cycle 127 (16/20)
Cached Engine  74 — Cycle 133 (17/20)
Cached Engine  31 — Cycle 188 (18/20)
Cached Engine  78 — Cycle 185 (19/20)
Cached Engine  46 — Cycle 205 (20/20)
✅ SHAP cache created for 20 engines.
ERMDSF ENGINE ANALYSIS COMPONENTS
✅ Model features detected: 24
✅ Engine-cycle mappings: 20631
✅ Cached fleet engines: 20
✅ Engine gauge function created.
✅ Local SHAP bar function cre

In [8]:
# ============================================================
# ENGINE ANALYSIS — SAFE PERCENTAGE FORMAT FIX
# ============================================================
def create_engine_summary_html(selected_row):
    """
    Create Engine Analysis summary cards using safe formatting.
    """
    engine_id = int(selected_row["engine_id"])
    cycle = int(selected_row["cycle"])
    predicted_rul = float(selected_row["predicted_RUL"])
    actual_rul = float(selected_row["actual_RUL"])
    lower_95 = float(selected_row["lower_95"])
    upper_95 = float(selected_row["upper_95"])
    failure_probability = float(
        selected_row["failure_probability"]
    )
    maintenance_priority = int(
        selected_row["maintenance_priority"]
    )
    risk = str(selected_row["risk_level"])
    interval_width = upper_95 - lower_95
    # Format all values before inserting them into HTML
    predicted_rul_text = f"{predicted_rul:.1f}"
    actual_rul_text = f"{actual_rul:.1f}"
    lower_text = f"{lower_95:.1f}"
    upper_text = f"{upper_95:.1f}"
    interval_width_text = f"{interval_width:.1f}"
    failure_probability_text = (
        f"{failure_probability * 100:.1f}%"
    )
    risk_class = (
        risk.lower()
        .strip()
        .replace(" ", "-")
    )
    risk_colors = {
        "Critical": COLORS.get("critical", "#D62828"),
        "High": COLORS.get("high", "#EF6C00"),
        "Medium": COLORS.get("warning", "#F9A825"),
        "Low": COLORS.get("success", "#2E8B3C")
    }
    risk_color = risk_colors.get(
        risk,
        COLORS.get("blue", "#176FAE")
    )
    return f"""
    <div class="engine-summary-grid">
        <div class="engine-metric-card">
            <div class="engine-metric-accent"></div>
            <div class="engine-metric-label">
                SELECTED ENGINE
            </div>
            <div class="engine-metric-value">
                Engine {engine_id}
            </div>
            <div class="engine-metric-note">
                Snapshot cycle: {cycle}
            </div>
        </div>
        <div class="engine-metric-card">
            <div class="engine-metric-accent"></div>
            <div class="engine-metric-label">
                PREDICTED RUL
            </div>
            <div class="engine-metric-value">
                {predicted_rul_text} cycles
            </div>
            <div class="engine-metric-note">
                Actual RUL: {actual_rul_text} cycles
            </div>
        </div>
        <div class="engine-metric-card">
            <div
                class="engine-metric-accent"
                style="background:{COLORS.get('high', '#EF6C00')};"
            ></div>
            <div class="engine-metric-label">
                FAILURE PROBABILITY
            </div>
            <div class="engine-metric-value">
                {failure_probability_text}
            </div>
            <div class="engine-metric-note">
                Estimated probability of RUL below
                {maintenance_threshold} cycles
            </div>
        </div>
        <div class="engine-metric-card">
            <div
                class="engine-metric-accent"
                style="background:{risk_color};"
            ></div>
            <div class="engine-metric-label">
                OPERATIONAL RISK
            </div>
            <div>
                <span class="risk-badge risk-{risk_class}">
                    {risk.upper()}
                </span>
            </div>
            <div class="engine-metric-note">
                Fleet priority: #{maintenance_priority}
            </div>
        </div>
    </div>
    <div class="panel-card">
        <div class="panel-title">
            Research Uncertainty Interval
        </div>
        <div style="
            color:{COLORS.get('text', '#082F4F')};
            font-size:22px;
            font-weight:800;
        ">
            {lower_text} – {upper_text} cycles
        </div>
        <div style="
            color:#7B8B97;
            font-size:10px;
            margin-top:5px;
            line-height:1.6;
        ">
            Interval width: {interval_width_text} cycles.
            The conformal interval provides uncertainty-aware
            supporting evidence and should not be interpreted
            as an automatic maintenance trigger.
        </div>
    </div>
    """
def create_engine_recommendation_html(selected_row):
    """
    Create maintenance recommendation using safe formatting.
    """
    recommendation = str(
        selected_row["recommendation"]
    )
    risk = str(
        selected_row["risk_level"]
    )
    predicted_rul = float(
        selected_row["predicted_RUL"]
    )
    failure_probability = float(
        selected_row["failure_probability"]
    )
    predicted_rul_text = f"{predicted_rul:.1f}"
    failure_probability_text = (
        f"{failure_probability * 100:.1f}%"
    )
    return f"""
    <div class="engine-recommendation">
        <div class="engine-recommendation-title">
            Maintenance Recommendation
        </div>
        <div class="engine-recommendation-text">
            {recommendation}
        </div>
        <div style="
            color:#71818E;
            font-size:10px;
            line-height:1.6;
            margin-top:9px;
        ">
            Supporting evidence:
            {risk} risk,
            predicted RUL {predicted_rul_text} cycles,
            failure probability {failure_probability_text}.
        </div>
    </div>
    """
def update_engine_analysis(engine_id):
    """
    Generate all outputs for one evaluation-fleet engine.
    """
    if engine_id is None:
        raise gr.Error(
            "Select an engine before loading the analysis."
        )
    engine_id = int(engine_id)
    if engine_id not in engine_cache:
        raise gr.Error(
            f"Engine {engine_id} is not available "
            "in the evaluation-fleet cache."
        )
    cached = engine_cache[engine_id]
    selected_row = cached["fleet_row"]
    contribution_table = cached["contribution_table"]
    explanation = cached["explanation"]
    summary_html = create_engine_summary_html(
        selected_row
    )
    gauge_figure = create_engine_health_gauge(
        float(selected_row["predicted_RUL"])
    )
    recommendation_html = (
        create_engine_recommendation_html(
            selected_row
        )
    )
    display_table = (
        contribution_table[
            [
                "Feature",
                "SHAP Value"
            ]
        ]
        .head(10)
        .copy()
    )
    display_table["SHAP Value"] = (
        pd.to_numeric(
            display_table["SHAP Value"],
            errors="coerce"
        )
        .round(4)
    )
    shap_bar_figure = create_local_shap_bar(
        contribution_table,
        top_n=10
    )
    waterfall_figure = create_local_waterfall(
        explanation
    )
    interpretation_html = (
        create_engine_interpretation_html(
            selected_row,
            contribution_table
        )
    )
    return (
        summary_html,
        gauge_figure,
        recommendation_html,
        display_table,
        shap_bar_figure,
        waterfall_figure,
        interpretation_html
    )
# ------------------------------------------------------------
# QUICK VALIDATION
# ------------------------------------------------------------
test_engine_id = int(
    sorted(engine_cache.keys())[0]
)
test_result = update_engine_analysis(
    test_engine_id
)
if len(test_result) != 7:
    raise RuntimeError(
        "Engine Analysis callback must return seven outputs."
    )
try:
    plt.close(test_result[5])
except Exception:
    plt.close("all")
print("=" * 65)
print("ENGINE ANALYSIS FORMAT FIX COMPLETED")
print("=" * 65)
print("✅ Percentage formatting corrected.")
print("✅ Summary-card function replaced.")
print("✅ Recommendation function replaced.")
print("✅ Engine Analysis callback replaced.")
print(f"✅ Tested successfully with Engine {test_engine_id}.")
print("=" * 65)

ENGINE ANALYSIS FORMAT FIX COMPLETED
✅ Percentage formatting corrected.
✅ Summary-card function replaced.
✅ Recommendation function replaced.
✅ Engine Analysis callback replaced.
✅ Tested successfully with Engine 1.


In [9]:
# ============================================================
# CELL 6 — MAINTENANCE PLANNER FUNCTIONS
# ============================================================
# ------------------------------------------------------------
# EMPTY PLAN HELPERS
# ------------------------------------------------------------
def empty_maintenance_table():
    """
    Return an empty planner table with the correct columns.
    """
    return pd.DataFrame(
        columns=[
            "Priority",
            "Engine",
            "Cycle",
            "Predicted RUL",
            "Failure Probability",
            "Risk",
            "Top RUL-Reducing Drivers",
            "Recommended Action"
        ]
    )
def empty_maintenance_chart(
    message="No maintenance plan available"
):
    """
    Return an empty Plotly figure for invalid or empty filters.
    """
    figure = go.Figure()
    figure.update_layout(
        title=dict(
            text=message,
            x=0.04,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis=dict(
            visible=False
        ),
        yaxis=dict(
            visible=False
        ),
        height=360,
        margin=dict(
            l=30,
            r=30,
            t=60,
            b=30
        )
    )
    return figure
# ------------------------------------------------------------
# PLAN SUMMARY HTML
# ------------------------------------------------------------
def create_planner_summary_html(
    selected_count,
    available_candidates,
    average_selected_rul,
    selected_high_critical,
    average_failure_probability,
    unselected_high_critical
):
    """
    Create maintenance-planner summary cards and
    capacity-status message.
    """
    if selected_count == 0:
        capacity_status_html = """
        <div class="warning-box" style="margin-top:12px;">
            No engines match the selected risk filters.
        </div>
        """
    elif unselected_high_critical > 0:
        capacity_status_html = f"""
        <div class="danger-box" style="margin-top:12px;">
            <b>Capacity warning:</b>
            {unselected_high_critical}
            High/Critical engine(s) remain outside
            the current inspection capacity.
        </div>
        """
    else:
        capacity_status_html = """
        <div class="success-box" style="margin-top:12px;">
            <b>Capacity adequate:</b>
            All matching High/Critical engines are included
            in the current inspection plan.
        </div>
        """
    return f"""
    <div class="panel-card">
        <div class="panel-title">
            Maintenance Plan Summary
        </div>
        <div style="
            display:grid;
            grid-template-columns:
                repeat(4, minmax(0, 1fr));
            gap:12px;
        ">
            <div style="
                background:#F6F9FC;
                border:1px solid #DCE5EC;
                border-radius:12px;
                padding:13px;
            ">
                <div style="
                    color:#70818E;
                    font-size:10px;
                    font-weight:800;
                ">
                    SELECTED ENGINES
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:23px;
                    font-weight:800;
                    margin-top:5px;
                ">
                    {selected_count}
                </div>
                <div style="
                    color:#8795A0;
                    font-size:10px;
                ">
                    From {available_candidates}
                    matching candidates
                </div>
            </div>
            <div style="
                background:#F6F9FC;
                border:1px solid #DCE5EC;
                border-radius:12px;
                padding:13px;
            ">
                <div style="
                    color:#70818E;
                    font-size:10px;
                    font-weight:800;
                ">
                    AVERAGE PREDICTED RUL
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:23px;
                    font-weight:800;
                    margin-top:5px;
                ">
                    {average_selected_rul:.1f}
                </div>
                <div style="
                    color:#8795A0;
                    font-size:10px;
                ">
                    Cycles
                </div>
            </div>
            <div style="
                background:#F6F9FC;
                border:1px solid #DCE5EC;
                border-radius:12px;
                padding:13px;
            ">
                <div style="
                    color:#70818E;
                    font-size:10px;
                    font-weight:800;
                ">
                    HIGH / CRITICAL SELECTED
                </div>
                <div style="
                    color:{COLORS['high']};
                    font-size:23px;
                    font-weight:800;
                    margin-top:5px;
                ">
                    {selected_high_critical}
                </div>
                <div style="
                    color:#8795A0;
                    font-size:10px;
                ">
                    Priority assets
                </div>
            </div>
            <div style="
                background:#F6F9FC;
                border:1px solid #DCE5EC;
                border-radius:12px;
                padding:13px;
            ">
                <div style="
                    color:#70818E;
                    font-size:10px;
                    font-weight:800;
                ">
                    AVG. FAILURE PROBABILITY
                </div>
                <div style="
                    color:{COLORS['critical']};
                    font-size:23px;
                    font-weight:800;
                    margin-top:5px;
                ">
                    {average_failure_probability:.1%}
                </div>
                <div style="
                    color:#8795A0;
                    font-size:10px;
                ">
                    Selected queue
                </div>
            </div>
        </div>
        {capacity_status_html}
    </div>
    """
# ------------------------------------------------------------
# PLANNER DISPLAY TABLE
# ------------------------------------------------------------
def create_planner_display_table(
    selected
):
    """
    Format the selected engines for dashboard display.
    """
    if selected.empty:
        return empty_maintenance_table()
    display_table = selected[
        [
            "maintenance_priority",
            "engine_id",
            "cycle",
            "predicted_RUL",
            "failure_probability",
            "risk_level",
            "RUL_reducing_drivers",
            "recommendation"
        ]
    ].copy()
    display_table.columns = [
        "Priority",
        "Engine",
        "Cycle",
        "Predicted RUL",
        "Failure Probability",
        "Risk",
        "Top RUL-Reducing Drivers",
        "Recommended Action"
    ]
    display_table[
        "Predicted RUL"
    ] = (
        display_table[
            "Predicted RUL"
        ]
        .round(1)
    )
    display_table[
        "Failure Probability"
    ] = (
        display_table[
            "Failure Probability"
        ]
        .map(
            lambda value:
            f"{value:.1%}"
        )
    )
    return display_table
# ------------------------------------------------------------
# PRIORITY CHART
# ------------------------------------------------------------
def create_planner_priority_chart(
    selected
):
    """
    Create a horizontal RUL chart for the selected
    inspection queue.
    """
    if selected.empty:
        return empty_maintenance_chart(
            "No engines match the selected filters"
        )
    chart_data = selected.copy()
    chart_data[
        "Engine Label"
    ] = (
        "Engine "
        +
        chart_data[
            "engine_id"
        ]
        .astype(int)
        .astype(str)
    )
    chart_data = (
        chart_data
        .sort_values(
            "maintenance_priority",
            ascending=False
        )
    )
    chart_colors = [
        RISK_COLORS.get(
            risk,
            COLORS["blue"]
        )
        for risk
        in chart_data[
            "risk_level"
        ]
    ]
    custom_data = chart_data[
        [
            "maintenance_priority",
            "failure_probability",
            "risk_level",
            "cycle"
        ]
    ].to_numpy()
    figure = go.Figure(
        go.Bar(
            x=chart_data[
                "predicted_RUL"
            ],
            y=chart_data[
                "Engine Label"
            ],
            orientation="h",
            marker_color=
                chart_colors,
            customdata=
                custom_data,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Priority: %{customdata[0]}<br>"
                "Cycle: %{customdata[3]}<br>"
                "Predicted RUL: %{x:.1f} cycles<br>"
                "Failure probability: "
                "%{customdata[1]:.1%}<br>"
                "Risk: %{customdata[2]}"
                "<extra></extra>"
            )
        )
    )
    figure.add_vline(
        x=critical_threshold,
        line_dash="dash",
        line_color=
            COLORS["critical"],
        line_width=2,
        annotation_text=(
            f"{critical_threshold}-cycle "
            "critical threshold"
        ),
        annotation_position=
            "top left"
    )
    figure.add_vline(
        x=maintenance_threshold,
        line_dash="dash",
        line_color=
            COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "maintenance threshold"
        ),
        annotation_position=
            "top right"
    )
    figure.add_vline(
        x=warning_threshold,
        line_dash="dash",
        line_color=
            COLORS["success"],
        line_width=2,
        annotation_text=(
            f"{warning_threshold}-cycle "
            "monitoring threshold"
        ),
        annotation_position=
            "top right"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Selected Inspection Queue — "
                "Predicted RUL"
            ),
            x=0.03,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Predicted Remaining Useful Life "
            "(cycles)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=90,
            r=25,
            t=60,
            b=50
        ),
        height=max(
            360,
            100 + len(selected) * 38
        ),
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor=(
            "rgba(0,0,0,0)"
        )
    )
    return figure
# ------------------------------------------------------------
# EXPORT MAINTENANCE PLAN
# ------------------------------------------------------------
def export_maintenance_plan(
    selected
):
    """
    Save the selected inspection queue as a CSV file.
    """
    output_directory = Path(
        "/content/ERMDSF_exports"
    )
    output_directory.mkdir(
        parents=True,
        exist_ok=True
    )
    timestamp = (
        datetime.now()
        .strftime(
            "%Y%m%d_%H%M%S"
        )
    )
    output_path = (
        output_directory
        /
        (
            "ERMDSF_maintenance_plan_"
            f"{timestamp}.csv"
        )
    )
    export_columns = [
        "maintenance_priority",
        "engine_id",
        "cycle",
        "actual_RUL",
        "predicted_RUL",
        "lower_95",
        "upper_95",
        "failure_probability",
        "risk_level",
        "RUL_reducing_drivers",
        "recommendation"
    ]
    export_table = selected[
        export_columns
    ].copy()
    export_table.columns = [
        "Maintenance Priority",
        "Engine ID",
        "Cycle",
        "Actual RUL",
        "Predicted RUL",
        "Lower Research Interval",
        "Upper Research Interval",
        "Failure Probability",
        "Risk Level",
        "RUL-Reducing SHAP Drivers",
        "Recommended Action"
    ]
    export_table.to_csv(
        output_path,
        index=False
    )
    return str(
        output_path
    )
# ------------------------------------------------------------
# MAIN PLANNER CALLBACK
# ------------------------------------------------------------
def build_maintenance_plan(
    inspection_capacity,
    selected_risks
):
    """
    Generate all outputs for the maintenance planner.
    """
    inspection_capacity = int(
        inspection_capacity
    )
    selected_risks = (
        selected_risks
        if selected_risks is not None
        else []
    )
    # No risk category selected
    if len(selected_risks) == 0:
        summary_html = """
        <div class="panel-card">
            <div class="panel-title">
                Maintenance Plan Summary
            </div>
            <div class="warning-box">
                Select at least one risk category
                to generate an inspection plan.
            </div>
        </div>
        """
        return (
            summary_html,
            empty_maintenance_table(),
            empty_maintenance_chart(
                "Select at least one risk category"
            ),
            None
        )
    # Filter fleet
    filtered = (
        fleet_dashboard[
            fleet_dashboard[
                "risk_level"
            ]
            .isin(
                selected_risks
            )
        ]
        .sort_values(
            "maintenance_priority"
        )
        .copy()
    )
    selected = (
        filtered
        .head(
            inspection_capacity
        )
        .copy()
    )
    selected_count = len(
        selected
    )
    available_candidates = len(
        filtered
    )
    high_critical_candidates = len(
        filtered[
            filtered[
                "risk_level"
            ]
            .isin(
                [
                    "Critical",
                    "High"
                ]
            )
        ]
    )
    selected_high_critical = len(
        selected[
            selected[
                "risk_level"
            ]
            .isin(
                [
                    "Critical",
                    "High"
                ]
            )
        ]
    )
    unselected_high_critical = max(
        high_critical_candidates
        -
        selected_high_critical,
        0
    )
    average_selected_rul = (
        float(
            selected[
                "predicted_RUL"
            ]
            .mean()
        )
        if selected_count > 0
        else 0.0
    )
    average_failure_probability = (
        float(
            selected[
                "failure_probability"
            ]
            .mean()
        )
        if selected_count > 0
        else 0.0
    )
    summary_html = (
        create_planner_summary_html(
            selected_count=
                selected_count,
            available_candidates=
                available_candidates,
            average_selected_rul=
                average_selected_rul,
            selected_high_critical=
                selected_high_critical,
            average_failure_probability=
                average_failure_probability,
            unselected_high_critical=
                unselected_high_critical
        )
    )
    display_table = (
        create_planner_display_table(
            selected
        )
    )
    priority_chart = (
        create_planner_priority_chart(
            selected
        )
    )
    export_path = (
        export_maintenance_plan(
            selected
        )
    )
    return (
        summary_html,
        display_table,
        priority_chart,
        export_path
    )
# ------------------------------------------------------------
# DEFAULT OUTPUTS
# ------------------------------------------------------------
default_planner_outputs = (
    build_maintenance_plan(
        inspection_capacity=5,
        selected_risks=[
            "Critical",
            "High",
            "Medium",
            "Low"
        ]
    )
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF MAINTENANCE PLANNER COMPONENTS")
print("=" * 65)
print(
    "✅ Capacity and risk-filter logic created."
)
print(
    "✅ Planner summary function created."
)
print(
    "✅ Inspection queue table created."
)
print(
    "✅ Priority chart function created."
)
print(
    "✅ CSV export function created."
)
print(
    "✅ Default five-engine plan generated."
)
print(
    f"Default export file: "
    f"{default_planner_outputs[3]}"
)
print("=" * 65)

ERMDSF MAINTENANCE PLANNER COMPONENTS
✅ Capacity and risk-filter logic created.
✅ Planner summary function created.
✅ Inspection queue table created.
✅ Priority chart function created.
✅ CSV export function created.
✅ Default five-engine plan generated.
Default export file: /content/ERMDSF_exports/ERMDSF_maintenance_plan_20261002_170604.csv


In [10]:
# ============================================================
# CELL 7 — FLEET HEALTH PAGE FUNCTIONS
# ============================================================
# ------------------------------------------------------------
# FLEET HEALTH KPI HTML
# ------------------------------------------------------------
def create_fleet_health_kpi_html():
    """
    Create fleet-wide health and prediction KPI cards.
    """
    mean_absolute_error_fleet = float(
        np.abs(
            fleet_dashboard["actual_RUL"]
            -
            fleet_dashboard["predicted_RUL"]
        ).mean()
    )
    mean_prediction_error = float(
        (
            fleet_dashboard["predicted_RUL"]
            -
            fleet_dashboard["actual_RUL"]
        ).mean()
    )
    within_10_cycles = int(
        (
            np.abs(
                fleet_dashboard["actual_RUL"]
                -
                fleet_dashboard["predicted_RUL"]
            )
            <= 10
        ).sum()
    )
    within_10_percentage = (
        within_10_cycles
        /
        fleet_size
    )
    return f"""
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                FLEET SIZE
            </div>
            <div class="kpi-value">
                {fleet_size}
            </div>
            <div class="kpi-note">
                Final-test fleet engines
            </div>
        </div>
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                MEAN ABSOLUTE ERROR
            </div>
            <div class="kpi-value">
                {mean_absolute_error_fleet:.1f}
                cycles
            </div>
            <div class="kpi-note">
                Fleet-snapshot prediction error
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="background:{COLORS['warning']};"
            >
            </div>
            <div class="kpi-label">
                MEAN PREDICTION BIAS
            </div>
            <div class="kpi-value">
                {mean_prediction_error:+.1f}
                cycles
            </div>
            <div class="kpi-note">
                Positive means overestimation
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="background:{COLORS['success']};"
            >
            </div>
            <div class="kpi-label">
                WITHIN ±10 CYCLES
            </div>
            <div class="kpi-value">
                {within_10_percentage:.0%}
            </div>
            <div class="kpi-note">
                {within_10_cycles} of
                {fleet_size} engines
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# FLEET RUL RANKING CHART
# ------------------------------------------------------------
def create_fleet_rul_ranking_chart():
    """
    Rank all engines by predicted RUL.
    """
    chart_data = (
        fleet_dashboard
        .sort_values(
            "predicted_RUL",
            ascending=True
        )
        .copy()
    )
    chart_data["Engine Label"] = (
        "Engine "
        +
        chart_data["engine_id"]
        .astype(int)
        .astype(str)
    )
    bar_colors = [
        RISK_COLORS.get(
            risk,
            COLORS["blue"]
        )
        for risk
        in chart_data["risk_level"]
    ]
    custom_data = chart_data[
        [
            "maintenance_priority",
            "failure_probability",
            "risk_level",
            "actual_RUL"
        ]
    ].to_numpy()
    figure = go.Figure(
        go.Bar(
            x=chart_data[
                "predicted_RUL"
            ],
            y=chart_data[
                "Engine Label"
            ],
            orientation="h",
            marker_color=
                bar_colors,
            customdata=
                custom_data,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Predicted RUL: %{x:.1f} cycles<br>"
                "Actual RUL: %{customdata[3]:.1f}<br>"
                "Fleet priority: %{customdata[0]}<br>"
                "Failure probability: "
                "%{customdata[1]:.1%}<br>"
                "Risk: %{customdata[2]}"
                "<extra></extra>"
            )
        )
    )
    figure.add_vline(
        x=critical_threshold,
        line_dash="dash",
        line_color=COLORS["critical"],
        line_width=2,
        annotation_text=(
            f"{critical_threshold}-cycle "
            "critical threshold"
        ),
        annotation_position="top left"
    )
    figure.add_vline(
        x=maintenance_threshold,
        line_dash="dash",
        line_color=COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "maintenance threshold"
        ),
        annotation_position="top right"
    )
    figure.add_vline(
        x=warning_threshold,
        line_dash="dash",
        line_color=COLORS["success"],
        line_width=2,
        annotation_text=(
            f"{warning_threshold}-cycle "
            "monitoring threshold"
        ),
        annotation_position="top right"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Fleet Ranking by Predicted RUL"
            ),
            x=0.03,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Predicted Remaining Useful Life "
            "(cycles)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=90,
            r=30,
            t=65,
            b=50
        ),
        height=720,
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor="rgba(0,0,0,0)"
    )
    return figure
# ------------------------------------------------------------
# ACTUAL VS PREDICTED RUL CHART
# ------------------------------------------------------------
def create_actual_vs_predicted_chart():
    """
    Compare actual and predicted RUL for all fleet engines.
    """
    chart_data = fleet_dashboard.copy()
    chart_data["Engine Label"] = (
        "Engine "
        +
        chart_data["engine_id"]
        .astype(int)
        .astype(str)
    )
    figure = go.Figure()
    for risk_level in risk_order:
        subset = chart_data[
            chart_data["risk_level"]
            ==
            risk_level
        ]
        if subset.empty:
            continue
        figure.add_trace(
            go.Scatter(
                x=subset[
                    "actual_RUL"
                ],
                y=subset[
                    "predicted_RUL"
                ],
                mode="markers",
                name=risk_level,
                marker=dict(
                    size=11,
                    color=RISK_COLORS[
                        risk_level
                    ],
                    line=dict(
                        color="white",
                        width=1
                    )
                ),
                customdata=subset[
                    [
                        "Engine Label",
                        "maintenance_priority",
                        "failure_probability"
                    ]
                ].to_numpy(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Actual RUL: %{x:.1f} cycles<br>"
                    "Predicted RUL: %{y:.1f} cycles<br>"
                    "Priority: %{customdata[1]}<br>"
                    "Failure probability: "
                    "%{customdata[2]:.1%}"
                    "<extra></extra>"
                )
            )
        )
    maximum_axis_value = float(
        max(
            fleet_dashboard[
                "actual_RUL"
            ].max(),
            fleet_dashboard[
                "predicted_RUL"
            ].max()
        )
    )
    axis_limit = max(
        125,
        np.ceil(
            maximum_axis_value / 10
        ) * 10
    )
    figure.add_trace(
        go.Scatter(
            x=[0, axis_limit],
            y=[0, axis_limit],
            mode="lines",
            name="Perfect prediction",
            line=dict(
                color="#7B8B97",
                width=2,
                dash="dash"
            ),
            hoverinfo="skip"
        )
    )
    figure.update_layout(
        title=dict(
            text="Actual vs Predicted RUL",
            x=0.04,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title="Actual RUL (cycles)",
        yaxis_title=(
            "Predicted RUL (cycles)"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=55,
            r=25,
            t=65,
            b=55
        ),
        height=440,
        legend=dict(
            orientation="h",
            y=-0.18,
            x=0.5,
            xanchor="center"
        )
    )
    figure.update_xaxes(
        range=[0, axis_limit],
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        range=[0, axis_limit],
        gridcolor="#E7EDF2",
        zeroline=False
    )
    return figure
# ------------------------------------------------------------
# FAILURE PROBABILITY VS RUL CHART
# ------------------------------------------------------------
def create_failure_probability_chart():
    """
    Show failure probability against predicted RUL.
    """
    chart_data = fleet_dashboard.copy()
    chart_data["Engine Label"] = (
        "Engine "
        +
        chart_data["engine_id"]
        .astype(int)
        .astype(str)
    )
    figure = go.Figure()
    for risk_level in risk_order:
        subset = chart_data[
            chart_data["risk_level"]
            ==
            risk_level
        ]
        if subset.empty:
            continue
        figure.add_trace(
            go.Scatter(
                x=subset[
                    "predicted_RUL"
                ],
                y=subset[
                    "failure_probability"
                ],
                mode="markers",
                name=risk_level,
                marker=dict(
                    size=(
                        11
                        +
                        (
                            21
                            -
                            subset[
                                "maintenance_priority"
                            ]
                        )
                        * 0.35
                    ),
                    color=RISK_COLORS[
                        risk_level
                    ],
                    line=dict(
                        color="white",
                        width=1
                    )
                ),
                customdata=subset[
                    [
                        "Engine Label",
                        "cycle",
                        "maintenance_priority",
                        "actual_RUL"
                    ]
                ].to_numpy(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Cycle: %{customdata[1]}<br>"
                    "Predicted RUL: %{x:.1f}<br>"
                    "Actual RUL: %{customdata[3]:.1f}<br>"
                    "Failure probability: %{y:.1%}<br>"
                    "Priority: %{customdata[2]}"
                    "<extra></extra>"
                )
            )
        )
    figure.add_vline(
        x=maintenance_threshold,
        line_dash="dash",
        line_color=COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "threshold"
        ),
        annotation_position="top right"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Failure Probability vs "
                "Predicted RUL"
            ),
            x=0.04,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Predicted RUL (cycles)"
        ),
        yaxis_title=(
            "Failure probability"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=60,
            r=25,
            t=65,
            b=55
        ),
        height=440,
        legend=dict(
            orientation="h",
            y=-0.18,
            x=0.5,
            xanchor="center"
        )
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        tickformat=".0%",
        range=[
            0,
            max(
                1.0,
                float(
                    fleet_dashboard[
                        "failure_probability"
                    ].max()
                )
                * 1.1
            )
        ],
        gridcolor="#E7EDF2",
        zeroline=False
    )
    return figure
# ------------------------------------------------------------
# UNCERTAINTY INTERVAL CHART
# ------------------------------------------------------------
def create_uncertainty_interval_chart():
    """
    Display predicted RUL with lower and upper
    conformal interval bounds.
    """
    chart_data = (
        fleet_dashboard
        .sort_values(
            "maintenance_priority",
            ascending=False
        )
        .copy()
    )
    chart_data["Engine Label"] = (
        "Engine "
        +
        chart_data["engine_id"]
        .astype(int)
        .astype(str)
    )
    lower_error = (
        chart_data["predicted_RUL"]
        -
        chart_data["lower_95"]
    ).clip(lower=0)
    upper_error = (
        chart_data["upper_95"]
        -
        chart_data["predicted_RUL"]
    ).clip(lower=0)
    point_colors = [
        RISK_COLORS.get(
            risk,
            COLORS["blue"]
        )
        for risk
        in chart_data["risk_level"]
    ]
    figure = go.Figure(
        go.Scatter(
            x=chart_data[
                "predicted_RUL"
            ],
            y=chart_data[
                "Engine Label"
            ],
            mode="markers",
            marker=dict(
                size=10,
                color=point_colors,
                line=dict(
                    color="white",
                    width=1
                )
            ),
            error_x=dict(
                type="data",
                symmetric=False,
                array=upper_error,
                arrayminus=lower_error,
                color="#8192A0",
                thickness=1.5,
                width=4
            ),
            customdata=chart_data[
                [
                    "lower_95",
                    "upper_95",
                    "actual_RUL",
                    "risk_level",
                    "maintenance_priority"
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Predicted RUL: %{x:.1f}<br>"
                "Research interval: "
                "%{customdata[0]:.1f}–"
                "%{customdata[1]:.1f}<br>"
                "Actual RUL: %{customdata[2]:.1f}<br>"
                "Risk: %{customdata[3]}<br>"
                "Priority: %{customdata[4]}"
                "<extra></extra>"
            )
        )
    )
    figure.add_vline(
        x=maintenance_threshold,
        line_dash="dash",
        line_color=COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "threshold"
        ),
        annotation_position="top right"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Fleet Prediction Uncertainty"
            ),
            x=0.03,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Predicted RUL and research interval "
            "(cycles)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=90,
            r=30,
            t=65,
            b=50
        ),
        height=720,
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor="rgba(0,0,0,0)"
    )
    return figure
# ------------------------------------------------------------
# FLEET HEALTH TABLE
# ------------------------------------------------------------
def create_fleet_health_table():
    """
    Create the full fleet-monitoring table.
    """
    table = fleet_dashboard[
        [
            "maintenance_priority",
            "engine_id",
            "cycle",
            "actual_RUL",
            "predicted_RUL",
            "lower_95",
            "upper_95",
            "failure_probability",
            "risk_level",
            "recommendation"
        ]
    ].copy()
    table.columns = [
        "Priority",
        "Engine",
        "Cycle",
        "Actual RUL",
        "Predicted RUL",
        "Lower Interval",
        "Upper Interval",
        "Failure Probability",
        "Risk",
        "Recommended Action"
    ]
    numeric_columns = [
        "Actual RUL",
        "Predicted RUL",
        "Lower Interval",
        "Upper Interval"
    ]
    for column in numeric_columns:
        table[column] = (
            table[column]
            .round(1)
        )
    table[
        "Failure Probability"
    ] = (
        table[
            "Failure Probability"
        ]
        .map(
            lambda value:
            f"{value:.1%}"
        )
    )
    return table
# ------------------------------------------------------------
# FLEET HEALTH INTERPRETATION
# ------------------------------------------------------------
def create_fleet_health_interpretation_html():
    """
    Create a concise interpretation of fleet-health results.
    """
    prediction_error = (
        fleet_dashboard[
            "predicted_RUL"
        ]
        -
        fleet_dashboard[
            "actual_RUL"
        ]
    )
    overestimated_count = int(
        (
            prediction_error > 5
        ).sum()
    )
    underestimated_count = int(
        (
            prediction_error < -5
        ).sum()
    )
    closely_aligned_count = (
        fleet_size
        -
        overestimated_count
        -
        underestimated_count
    )
    widest_interval_row = (
        fleet_dashboard
        .assign(
            interval_width=(
                fleet_dashboard[
                    "upper_95"
                ]
                -
                fleet_dashboard[
                    "lower_95"
                ]
            )
        )
        .sort_values(
            "interval_width",
            ascending=False
        )
        .iloc[0]
    )
    return f"""
    <div class="panel-card">
        <div class="panel-title">
            Fleet Health Interpretation
        </div>
        <div style=
            "color:#526875;
            font-size:12px;
            line-height:1.7;"
        >
            <b>{high_critical_count}</b>
            of the
            <b>{fleet_size}</b>
            monitored engines are classified as
            High or Critical risk.
            <br><br>
            The model overestimates RUL by more than
            five cycles for
            <b>{overestimated_count}</b>
            engine(s), while it provides a conservative
            underestimate for
            <b>{underestimated_count}</b>
            engine(s).
            <br><br>
            Predictions are within ±5 cycles for
            <b>{closely_aligned_count}</b>
            engine(s).
            <br><br>
            The widest conformal interval belongs to
            <b>
                Engine
                {int(widest_interval_row['engine_id'])}
            </b>,
            with an interval width of
            <b>
                {widest_interval_row['interval_width']:.1f}
                cycles
            </b>.
            This confirms that prediction uncertainty
            should be treated as supporting evidence,
            alongside SHAP explanations and engineering
            judgement.
        </div>
    </div>
    """
# ------------------------------------------------------------
# BUILD FLEET PAGE OBJECTS
# ------------------------------------------------------------
fleet_health_kpi_html = (
    create_fleet_health_kpi_html()
)
fleet_rul_ranking_chart = (
    create_fleet_rul_ranking_chart()
)
actual_vs_predicted_chart = (
    create_actual_vs_predicted_chart()
)
failure_probability_chart = (
    create_failure_probability_chart()
)
uncertainty_interval_chart = (
    create_uncertainty_interval_chart()
)
fleet_health_table = (
    create_fleet_health_table()
)
fleet_health_interpretation_html = (
    create_fleet_health_interpretation_html()
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF FLEET HEALTH COMPONENTS")
print("=" * 65)
print("✅ Fleet-health KPI cards created.")
print("✅ Fleet RUL ranking chart created.")
print("✅ Actual-vs-predicted chart created.")
print("✅ Failure-probability chart created.")
print("✅ Uncertainty-interval chart created.")
print("✅ Complete fleet table created.")
print("✅ Fleet interpretation created.")
print("=" * 65)

ERMDSF FLEET HEALTH COMPONENTS
✅ Fleet-health KPI cards created.
✅ Fleet RUL ranking chart created.
✅ Actual-vs-predicted chart created.
✅ Failure-probability chart created.
✅ Uncertainty-interval chart created.
✅ Complete fleet table created.
✅ Fleet interpretation created.


In [11]:
# ============================================================
# CELL 8 — EXPLAINABILITY CENTER FUNCTIONS
# ============================================================
# ------------------------------------------------------------
# COMBINE LOCAL SHAP VALUES ACROSS FLEET SNAPSHOTS
# ------------------------------------------------------------
fleet_shap_records = []
for engine_id, cached_data in engine_cache.items():
    contribution_table = (
        cached_data[
            "contribution_table"
        ]
        .copy()
    )
    contribution_table[
        "Engine"
    ] = int(
        engine_id
    )
    fleet_shap_records.append(
        contribution_table
    )
fleet_shap_long = pd.concat(
    fleet_shap_records,
    ignore_index=True
)
# ------------------------------------------------------------
# GLOBAL SHAP RANKING
# ------------------------------------------------------------
global_shap_ranking = (
    fleet_shap_long
    .groupby(
        "Feature",
        as_index=False
    )
    .agg(
        Mean_Absolute_SHAP=(
            "Absolute Contribution",
            "mean"
        ),
        Mean_Signed_SHAP=(
            "SHAP Value",
            "mean"
        ),
        SHAP_Standard_Deviation=(
            "SHAP Value",
            "std"
        )
    )
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)
global_shap_ranking[
    "Rank"
] = (
    np.arange(
        1,
        len(global_shap_ranking) + 1
    )
)
global_shap_ranking = (
    global_shap_ranking[
        [
            "Rank",
            "Feature",
            "Mean_Absolute_SHAP",
            "Mean_Signed_SHAP",
            "SHAP_Standard_Deviation"
        ]
    ]
)
# ------------------------------------------------------------
# DISPLAY VERSION OF GLOBAL RANKING
# ------------------------------------------------------------
global_shap_table = (
    global_shap_ranking
    .head(15)
    .copy()
)
for column in [
    "Mean_Absolute_SHAP",
    "Mean_Signed_SHAP",
    "SHAP_Standard_Deviation"
]:
    global_shap_table[
        column
    ] = (
        global_shap_table[
            column
        ]
        .round(4)
    )
global_shap_table.columns = [
    "Rank",
    "Feature",
    "Mean |SHAP|",
    "Mean Signed SHAP",
    "SHAP Standard Deviation"
]
# ------------------------------------------------------------
# GLOBAL SHAP BAR CHART
# ------------------------------------------------------------
def create_global_shap_bar(
    top_n=15
):
    """
    Create global mean-absolute-SHAP ranking chart.
    """
    chart_data = (
        global_shap_ranking
        .head(top_n)
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=True
        )
        .copy()
    )
    figure = go.Figure(
        go.Bar(
            x=chart_data[
                "Mean_Absolute_SHAP"
            ],
            y=chart_data[
                "Feature"
            ],
            orientation="h",
            marker=dict(
                color=chart_data[
                    "Mean_Absolute_SHAP"
                ],
                colorscale=[
                    [0.0, "#BFD8EA"],
                    [1.0, COLORS["blue"]]
                ],
                line=dict(
                    color="white",
                    width=0.8
                )
            ),
            customdata=chart_data[
                [
                    "Rank",
                    "Mean_Signed_SHAP",
                    "SHAP_Standard_Deviation"
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Rank: %{customdata[0]}<br>"
                "Mean |SHAP|: %{x:.4f}<br>"
                "Mean signed SHAP: "
                "%{customdata[1]:+.4f}<br>"
                "SHAP SD: %{customdata[2]:.4f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "Global XGBoost SHAP "
                "Feature Importance"
            ),
            x=0.04,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Mean absolute SHAP contribution "
            "(cycles)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=95,
            r=25,
            t=65,
            b=55
        ),
        height=520,
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor="rgba(0,0,0,0)"
    )
    return figure
# ------------------------------------------------------------
# SENSOR CONTRIBUTION HEATMAP
# ------------------------------------------------------------
def create_shap_heatmap(
    top_n_features=10
):
    """
    Show signed SHAP contributions for the most
    important sensors across all fleet engines.
    """
    top_features = (
        global_shap_ranking
        .head(top_n_features)[
            "Feature"
        ]
        .tolist()
    )
    heatmap_data = pd.DataFrame(
        index=sorted(
            engine_cache.keys()
        ),
        columns=top_features,
        dtype=float
    )
    for engine_id, cached_data in engine_cache.items():
        local_table = (
            cached_data[
                "contribution_table"
            ]
            .set_index(
                "Feature"
            )
        )
        for feature in top_features:
            heatmap_data.loc[
                engine_id,
                feature
            ] = float(
                local_table.loc[
                    feature,
                    "SHAP Value"
                ]
            )
    heatmap_data.index = [
        f"Engine {int(engine_id)}"
        for engine_id
        in heatmap_data.index
    ]
    maximum_absolute_value = float(
        np.nanmax(
            np.abs(
                heatmap_data.to_numpy(
                    dtype=float
                )
            )
        )
    )
    figure = go.Figure(
        go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale=[
                [0.0, COLORS["critical"]],
                [0.5, "#F7F9FB"],
                [1.0, COLORS["blue"]]
            ],
            zmin=(
                -maximum_absolute_value
            ),
            zmax=(
                maximum_absolute_value
            ),
            colorbar=dict(
                title=(
                    "SHAP<br>"
                    "cycles"
                )
            ),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Feature: %{x}<br>"
                "SHAP value: %{z:+.3f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "Fleet-Level Signed SHAP "
                "Contribution Heatmap"
            ),
            x=0.03,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=(
            "Sensor feature"
        ),
        yaxis_title=(
            "Fleet engine"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=90,
            r=30,
            t=70,
            b=70
        ),
        height=650
    )
    figure.update_xaxes(
        tickangle=-35
    )
    return figure
# ------------------------------------------------------------
# CROSS-MODEL SENSOR CONSISTENCY TABLE
# ------------------------------------------------------------
# Rankings obtained from the completed thesis experiments.
linear_regression_top10 = [
    "sensor_11",
    "sensor_12",
    "sensor_4",
    "sensor_7",
    "sensor_9",
    "sensor_15",
    "sensor_14",
    "sensor_20",
    "sensor_21",
    "sensor_2"
]
random_forest_top10 = [
    "sensor_11",
    "sensor_4",
    "sensor_12",
    "sensor_7",
    "sensor_15",
    "sensor_21",
    "sensor_9",
    "sensor_14",
    "sensor_2",
    "sensor_20"
]
xgboost_top10 = (
    global_shap_ranking
    .head(10)[
        "Feature"
    ]
    .tolist()
)
all_consistency_features = sorted(
    set(
        linear_regression_top10
        +
        random_forest_top10
        +
        xgboost_top10
    )
)
sensor_consistency_rows = []
for feature in all_consistency_features:
    present_lr = (
        feature
        in linear_regression_top10
    )
    present_rf = (
        feature
        in random_forest_top10
    )
    present_xgb = (
        feature
        in xgboost_top10
    )
    model_count = sum(
        [
            present_lr,
            present_rf,
            present_xgb
        ]
    )
    if model_count == 3:
        consistency_label = (
            "Very High"
        )
    elif model_count == 2:
        consistency_label = (
            "High"
        )
    elif model_count == 1:
        consistency_label = (
            "Model-Specific"
        )
    else:
        consistency_label = (
            "Not Present"
        )
    sensor_consistency_rows.append({
        "Sensor": feature,
        "Linear Regression":
            "✓" if present_lr else "—",
        "Random Forest":
            "✓" if present_rf else "—",
        "XGBoost":
            "✓" if present_xgb else "—",
        "Models in Top 10":
            model_count,
        "Consistency":
            consistency_label
    })
sensor_consistency_table = (
    pd.DataFrame(
        sensor_consistency_rows
    )
    .sort_values(
        [
            "Models in Top 10",
            "Sensor"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)
# ------------------------------------------------------------
# GLOBAL EXPLAINABILITY KPI HTML
# ------------------------------------------------------------
def create_explainability_kpi_html():
    """
    Create summary cards for explanation quality.
    """
    top_feature = str(
        global_shap_ranking
        .iloc[0]["Feature"]
    )
    top_feature_importance = float(
        global_shap_ranking
        .iloc[0][
            "Mean_Absolute_SHAP"
        ]
    )
    consistent_sensor_count = int(
        (
            sensor_consistency_table[
                "Models in Top 10"
            ]
            ==
            3
        )
        .sum()
    )
    return f"""
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                DOMINANT SENSOR
            </div>
            <div class="kpi-value">
                {top_feature}
            </div>
            <div class="kpi-note">
                Mean |SHAP|:
                {top_feature_importance:.2f}
                cycles
            </div>
        </div>
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                CROSS-MODEL SPEARMAN
            </div>
            <div class="kpi-value">
                {cross_model_spearman:.3f}
            </div>
            <div class="kpi-note">
                Entire-ranking agreement
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['warning']};
                "
            >
            </div>
            <div class="kpi-label">
                TOP-5 JACCARD
            </div>
            <div class="kpi-value">
                {cross_model_jaccard:.3f}
            </div>
            <div class="kpi-note">
                Most-important-sensor overlap
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['success']};
                "
            >
            </div>
            <div class="kpi-label">
                CONSISTENT TOP-10 SENSORS
            </div>
            <div class="kpi-value">
                {consistent_sensor_count}
            </div>
            <div class="kpi-note">
                Present in all three models
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# GLOBAL EXPLAINABILITY INTERPRETATION
# ------------------------------------------------------------
def create_global_explainability_html():
    """
    Create research interpretation for global SHAP results.
    """
    top_five_features = (
        global_shap_ranking
        .head(5)[
            "Feature"
        ]
        .tolist()
    )
    top_five_text = ", ".join(
        top_five_features
    )
    consistent_features = (
        sensor_consistency_table[
            sensor_consistency_table[
                "Models in Top 10"
            ]
            ==
            3
        ][
            "Sensor"
        ]
        .tolist()
    )
    consistent_text = ", ".join(
        consistent_features
    )
    return f"""
    <div class="panel-card">
        <div class="panel-title">
            Explainability Interpretation
        </div>
        <div style="
            color:#526875;
            font-size:12px;
            line-height:1.7;
        ">
            The most influential XGBoost sensors
            across the fleet snapshots are
            <b>
                {top_five_text}
            </b>.
            <br><br>
            The cross-model Spearman correlation of
            <b>
                {cross_model_spearman:.3f}
            </b>
            indicates very high agreement across the
            complete Linear Regression, Random Forest
            and XGBoost rankings.
            <br><br>
            The Top-5 Jaccard similarity of
            <b>
                {cross_model_jaccard:.3f}
            </b>
            shows that the models share most, but not
            necessarily all, of their highest-priority
            sensors.
            <br><br>
            Sensors consistently appearing in the
            Top-10 rankings of all three models include
            <b>
                {consistent_text}
            </b>.
            <br><br>
            The remaining differences are expected
            because Linear Regression represents global
            linear effects, Random Forest captures
            nonlinear averaging across trees, and
            XGBoost models sequential nonlinear
            interactions and residual structure.
        </div>
    </div>
    """
# ------------------------------------------------------------
# LOCAL EXPLAINABILITY CALLBACK
# ------------------------------------------------------------
def update_explainability_engine(
    engine_id
):
    """
    Return local explainability outputs for the
    Explainability Center.
    """
    engine_id = int(
        engine_id
    )
    if engine_id not in engine_cache:
        raise gr.Error(
            f"Engine {engine_id} is not available."
        )
    cached = engine_cache[
        engine_id
    ]
    selected_row = cached[
        "fleet_row"
    ]
    contribution_table = cached[
        "contribution_table"
    ]
    explanation = cached[
        "explanation"
    ]
    display_table = (
        contribution_table[
            [
                "Feature",
                "SHAP Value",
                "Absolute Contribution"
            ]
        ]
        .head(15)
        .copy()
    )
    display_table[
        "SHAP Value"
    ] = (
        display_table[
            "SHAP Value"
        ]
        .round(4)
    )
    display_table[
        "Absolute Contribution"
    ] = (
        display_table[
            "Absolute Contribution"
        ]
        .round(4)
    )
    display_table.columns = [
        "Feature",
        "Signed SHAP",
        "Absolute SHAP"
    ]
    local_bar = (
        create_local_shap_bar(
            contribution_table,
            top_n=12
        )
    )
    local_waterfall = (
        create_local_waterfall(
            explanation
        )
    )
    local_interpretation = (
        create_engine_interpretation_html(
            selected_row,
            contribution_table
        )
    )
    return (
        display_table,
        local_bar,
        local_waterfall,
        local_interpretation
    )
# ------------------------------------------------------------
# BUILD EXPLAINABILITY OBJECTS
# ------------------------------------------------------------
explainability_kpi_html = (
    create_explainability_kpi_html()
)
global_shap_bar_chart = (
    create_global_shap_bar(
        top_n=15
    )
)
fleet_shap_heatmap = (
    create_shap_heatmap(
        top_n_features=10
    )
)
global_explainability_html = (
    create_global_explainability_html()
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF EXPLAINABILITY CENTER COMPONENTS")
print("=" * 65)
print(
    f"✅ Global SHAP features ranked: "
    f"{len(global_shap_ranking)}"
)
print(
    "✅ Global SHAP bar chart created."
)
print(
    "✅ Fleet SHAP heatmap created."
)
print(
    "✅ Cross-model consistency table created."
)
print(
    "✅ Explainability KPI cards created."
)
print(
    "✅ Global interpretation created."
)
print(
    "✅ Local explainability callback created."
)
print("=" * 65)

ERMDSF EXPLAINABILITY CENTER COMPONENTS
✅ Global SHAP features ranked: 24
✅ Global SHAP bar chart created.
✅ Fleet SHAP heatmap created.
✅ Cross-model consistency table created.
✅ Explainability KPI cards created.
✅ Global interpretation created.
✅ Local explainability callback created.


In [12]:
# ============================================================
# ERMDSF — RELIABILITY PAGE
# FINAL FOUR-MODEL VERSION
#
# Models:
#   1. Linear Regression
#   2. Random Forest
#   3. XGBoost
#   4. Support Vector Regression (SVR)
#
# Includes:
#   - Predictive performance comparison
#   - Four-model cross-model SHAP stability
#   - Random train/test split SHAP stability
#   - Spearman + Jaccard interpretation
# ============================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gradio as gr
# ============================================================
# 1. VALIDATED FOUR-MODEL RESULTS
# ============================================================
# ------------------------------------------------------------
# Baseline predictive performance
# ------------------------------------------------------------
reliability_model_performance = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ],
    "Model Family": [
        "Linear",
        "Tree-based Bagging",
        "Tree-based Boosting",
        "Kernel-based Nonlinear"
    ],
    "RMSE": [
        20.41,
        17.04,
        16.88,
        17.6783
    ]
})
# ------------------------------------------------------------
# Random engine-level train/test split SHAP stability
# ------------------------------------------------------------
reliability_split_stability = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ],
    "Spearman Mean": [
        0.9883,
        0.9944,
        0.9837,
        0.9565
    ],
    "Spearman SD": [
        0.0068,
        0.0035,
        0.0080,
        0.0187
    ],
    "Top-5 Jaccard Mean": [
        0.8000,
        0.8000,
        1.0000,
        0.8667
    ],
    "Top-5 Jaccard SD": [
        0.1633,
        0.1633,
        0.0000,
        0.1721
    ]
})
# ------------------------------------------------------------
# Four-model CROSS-MODEL results
#
# LR vs RF
# LR vs XGB
# LR vs SVR
# RF vs XGB
# RF vs SVR
# XGB vs SVR
# ------------------------------------------------------------
FOUR_MODEL_CROSS_SPEARMAN_MEAN = 0.9758
FOUR_MODEL_CROSS_SPEARMAN_SD = 0.0146
FOUR_MODEL_CROSS_JACCARD_MEAN = 0.8333
FOUR_MODEL_CROSS_JACCARD_SD = 0.1826
# ------------------------------------------------------------
# Pairwise cross-model Spearman
# ------------------------------------------------------------
cross_model_spearman_4 = pd.DataFrame(
    [
        [1.0000, 0.9843, 0.9730, 0.9504],
        [0.9843, 1.0000, 0.9896, 0.9704],
        [0.9730, 0.9896, 1.0000, 0.9870],
        [0.9504, 0.9704, 0.9870, 1.0000]
    ],
    index=[
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ],
    columns=[
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ]
)
# ------------------------------------------------------------
# Pairwise cross-model Top-5 Jaccard
# ------------------------------------------------------------
cross_model_jaccard_4 = pd.DataFrame(
    [
        [1.0000, 0.6667, 0.6667, 0.6667],
        [0.6667, 1.0000, 1.0000, 1.0000],
        [0.6667, 1.0000, 1.0000, 1.0000],
        [0.6667, 1.0000, 1.0000, 1.0000]
    ],
    index=[
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ],
    columns=[
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ]
)
# ============================================================
# 2. DISPLAY TABLES
# ============================================================
reliability_model_performance_display = (
    reliability_model_performance.copy()
)
reliability_model_performance_display[
    "RMSE"
] = (
    reliability_model_performance_display[
        "RMSE"
    ]
    .round(4)
)
reliability_split_display = pd.DataFrame({
    "Model":
        reliability_split_stability["Model"],
    "Spearman ρ (Mean ± SD)":
        reliability_split_stability.apply(
            lambda row:
            (
                f'{row["Spearman Mean"]:.4f} '
                f'± {row["Spearman SD"]:.4f}'
            ),
            axis=1
        ),
    "Top-5 Jaccard (Mean ± SD)":
        reliability_split_stability.apply(
            lambda row:
            (
                f'{row["Top-5 Jaccard Mean"]:.4f} '
                f'± {row["Top-5 Jaccard SD"]:.4f}'
            ),
            axis=1
        )
})
# ============================================================
# 3. KPI HTML
# ============================================================
def create_reliability_kpi_html():
    return f"""
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                MODELS EVALUATED
            </div>
            <div class="kpi-value">
                4
            </div>
            <div class="kpi-note">
                Linear, bagging, boosting and
                kernel-based regression
            </div>
        </div>
        <div class="kpi-card">
            <div class="kpi-accent">
            </div>
            <div class="kpi-label">
                CROSS-MODEL SPEARMAN
            </div>
            <div class="kpi-value">
                {FOUR_MODEL_CROSS_SPEARMAN_MEAN:.3f}
            </div>
            <div class="kpi-note">
                ± {FOUR_MODEL_CROSS_SPEARMAN_SD:.3f}
                full-ranking agreement
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['warning']};
                "
            >
            </div>
            <div class="kpi-label">
                CROSS-MODEL TOP-5 JACCARD
            </div>
            <div class="kpi-value">
                {FOUR_MODEL_CROSS_JACCARD_MEAN:.3f}
            </div>
            <div class="kpi-note">
                ± {FOUR_MODEL_CROSS_JACCARD_SD:.3f}
                important-sensor overlap
            </div>
        </div>
        <div class="kpi-card">
            <div
                class="kpi-accent"
                style="
                    background:
                    {COLORS['success']};
                "
            >
            </div>
            <div class="kpi-label">
                SVR BASELINE RMSE
            </div>
            <div class="kpi-value">
                17.68
            </div>
            <div class="kpi-note">
                Comparable to RF and XGBoost
            </div>
        </div>
    </div>
    """
# ============================================================
# 4. MODEL PERFORMANCE CHART
# ============================================================
def create_reliability_model_chart():
    chart_data = (
        reliability_model_performance
        .sort_values(
            "RMSE",
            ascending=False
        )
        .copy()
    )
    figure = go.Figure(
        go.Bar(
            x=chart_data[
                "RMSE"
            ],
            y=chart_data[
                "Model"
            ],
            orientation="h",
            text=chart_data[
                "RMSE"
            ].map(
                lambda value:
                f"{value:.2f}"
            ),
            textposition="outside",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "RMSE: %{x:.4f} cycles"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "Predictive Performance "
                "Across Four Model Families"
            ),
            x=0.03
        ),
        xaxis_title=(
            "RMSE (cycles — lower is better)"
        ),
        yaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=430,
        margin=dict(
            l=130,
            r=50,
            t=70,
            b=60
        ),
        showlegend=False
    )
    figure.update_xaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    figure.update_yaxes(
        gridcolor="rgba(0,0,0,0)"
    )
    return figure
# ============================================================
# 5. CROSS-MODEL SPEARMAN HEATMAP
# ============================================================
def create_four_model_spearman_heatmap():
    models = list(
        cross_model_spearman_4.index
    )
    figure = go.Figure(
        go.Heatmap(
            z=cross_model_spearman_4.values,
            x=models,
            y=models,
            zmin=0.90,
            zmax=1.00,
            colorscale="Blues",
            text=np.round(
                cross_model_spearman_4.values,
                3
            ),
            texttemplate="%{text}",
            colorbar=dict(
                title="Spearman ρ"
            ),
            hovertemplate=(
                "%{y} vs %{x}<br>"
                "Spearman ρ: %{z:.4f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "Cross-Model SHAP Ranking Agreement "
                "— Spearman Correlation"
            ),
            x=0.03
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=520,
        margin=dict(
            l=130,
            r=40,
            t=75,
            b=110
        )
    )
    figure.update_xaxes(
        tickangle=-30
    )
    return figure
# ============================================================
# 6. CROSS-MODEL JACCARD HEATMAP
# ============================================================
def create_four_model_jaccard_heatmap():
    models = list(
        cross_model_jaccard_4.index
    )
    figure = go.Figure(
        go.Heatmap(
            z=cross_model_jaccard_4.values,
            x=models,
            y=models,
            zmin=0.0,
            zmax=1.0,
            colorscale="Blues",
            text=np.round(
                cross_model_jaccard_4.values,
                3
            ),
            texttemplate="%{text}",
            colorbar=dict(
                title="Jaccard"
            ),
            hovertemplate=(
                "%{y} vs %{x}<br>"
                "Top-5 Jaccard: %{z:.4f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "Cross-Model Top-5 Sensor Agreement "
                "— Jaccard Similarity"
            ),
            x=0.03
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=520,
        margin=dict(
            l=130,
            r=40,
            t=75,
            b=110
        )
    )
    figure.update_xaxes(
        tickangle=-30
    )
    return figure
# ============================================================
# 7. RANDOM-SPLIT STABILITY CHART
# ============================================================
def create_random_split_stability_chart():
    figure = go.Figure()
    figure.add_trace(
        go.Bar(
            name="Spearman",
            x=reliability_split_stability[
                "Model"
            ],
            y=reliability_split_stability[
                "Spearman Mean"
            ],
            error_y=dict(
                type="data",
                array=reliability_split_stability[
                    "Spearman SD"
                ],
                visible=True
            ),
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Spearman: %{y:.4f}"
                "<extra></extra>"
            )
        )
    )
    figure.add_trace(
        go.Bar(
            name="Top-5 Jaccard",
            x=reliability_split_stability[
                "Model"
            ],
            y=reliability_split_stability[
                "Top-5 Jaccard Mean"
            ],
            error_y=dict(
                type="data",
                array=reliability_split_stability[
                    "Top-5 Jaccard SD"
                ],
                visible=True
            ),
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Jaccard: %{y:.4f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=dict(
            text=(
                "SHAP Stability Under Different "
                "Engine-Level Train/Test Splits"
            ),
            x=0.03
        ),
        barmode="group",
        yaxis=dict(
            title="Stability score",
            range=[0, 1.08]
        ),
        xaxis_title=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=500,
        margin=dict(
            l=65,
            r=30,
            t=75,
            b=100
        ),
        legend=dict(
            orientation="h",
            y=1.10,
            x=0
        )
    )
    figure.update_yaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    return figure
# ============================================================
# 8. INTERPRETATION HTML
# ============================================================
def create_reliability_interpretation_html():
    return f"""
    <div class="panel-card">
        <div class="panel-title">
            Reliability Interpretation
        </div>
        <div style="
            color:#526875;
            font-size:13px;
            line-height:1.75;
        ">
            Four heterogeneous regression approaches
            were evaluated:
            <b>
                Linear Regression,
                Random Forest,
                XGBoost and
                Support Vector Regression (SVR).
            </b>
            <br><br>
            SVR provides an additional
            <b>kernel-based nonlinear model</b>,
            extending the comparison beyond the
            linear and tree-based algorithms.
            Its baseline RMSE of
            <b>
                17.6783 cycles
            </b>
            is close to Random Forest
            (17.04) and XGBoost (16.88),
            demonstrating comparable predictive
            capability despite the different
            model architecture.
            <br><br>
            Across all six unique model-pair
            comparisons, the complete SHAP feature
            rankings produced a mean Spearman
            correlation of
            <b>
                {FOUR_MODEL_CROSS_SPEARMAN_MEAN:.4f}
                ±
                {FOUR_MODEL_CROSS_SPEARMAN_SD:.4f}.
            </b>
            This indicates very high agreement in
            the global ordering of sensor importance.
            <br><br>
            The mean Top-5 Jaccard similarity was
            <b>
                {FOUR_MODEL_CROSS_JACCARD_MEAN:.4f}
                ±
                {FOUR_MODEL_CROSS_JACCARD_SD:.4f}.
            </b>
            Random Forest, XGBoost and SVR identified
            exactly the same Top-5 sensor set,
            resulting in pairwise Jaccard values
            of 1.000 among these three models.
            <br><br>
            Under different engine-level train/test
            partitions, all four models also retained
            high SHAP ranking stability.
            Random Forest achieved the highest mean
            Spearman stability
            (<b>0.9944</b>), followed by Linear
            Regression (<b>0.9883</b>), XGBoost
            (<b>0.9837</b>) and SVR
            (<b>0.9565</b>).
            <br><br>
            XGBoost achieved perfect Top-5 stability
            across the evaluated splits
            (<b>Jaccard = 1.0000</b>).
            SVR achieved a mean Top-5 Jaccard
            similarity of
            <b>
                0.8667 ± 0.1721,
            </b>
            showing that its most influential
            degradation-related sensors also remained
            largely stable when the composition of
            the training and test engine populations
            changed.
            <br><br>
            Overall, the results indicate that the
            principal sensor patterns identified by
            SHAP are not limited to one specific
            machine-learning architecture.
        </div>
    </div>
    """
# ============================================================
# 9. CREATE RELIABILITY OBJECTS
# ============================================================
reliability_kpi_html = (
    create_reliability_kpi_html()
)
reliability_model_chart = (
    create_reliability_model_chart()
)
reliability_spearman_heatmap = (
    create_four_model_spearman_heatmap()
)
reliability_jaccard_heatmap = (
    create_four_model_jaccard_heatmap()
)
reliability_split_chart = (
    create_random_split_stability_chart()
)
reliability_interpretation_html = (
    create_reliability_interpretation_html()
)
# ============================================================
# 10. FINAL RELIABILITY PAGE BUILDER
# ============================================================
def build_reliability_page():
    """
    Build ERMDSF Reliability page.
    Four-model version:
    LR + RF + XGBoost + SVR.
    """
    with gr.Column(
        visible=False,
        elem_classes=["page-shell"]
    ) as page:
        # ----------------------------------------------------
        # PAGE HEADER
        # ----------------------------------------------------
        gr.HTML(
            """
            <div class="page-header-card">
                <div class="page-eyebrow">
                    MODEL VALIDATION
                </div>
                <div class="page-title">
                    Reliability & Explanation Stability
                </div>
                <div class="page-subtitle">
                    Comparative assessment of predictive
                    performance and SHAP explanation stability
                    across four heterogeneous machine-learning
                    architectures.
                </div>
            </div>
            """
        )
        # ----------------------------------------------------
        # KPI CARDS
        # ----------------------------------------------------
        gr.HTML(
            reliability_kpi_html
        )
        # ----------------------------------------------------
        # MODEL PERFORMANCE
        # ----------------------------------------------------
        gr.HTML(
            """
            <div class="section-heading">
                Predictive Model Comparison
            </div>
            """
        )
        with gr.Row():
            with gr.Column(scale=2):
                gr.Dataframe(
                    value=(
                        reliability_model_performance_display
                    ),
                    headers=[
                        "Model",
                        "Model Family",
                        "RMSE"
                    ],
                    datatype=[
                        "str",
                        "str",
                        "number"
                    ],
                    interactive=False,
                    wrap=True,
                    label=(
                        "Baseline Predictive Performance"
                    )
                )
            with gr.Column(scale=3):
                gr.Plot(
                    value=reliability_model_chart,
                    label=(
                        "RMSE Comparison"
                    )
                )
        # ----------------------------------------------------
        # CROSS-MODEL EXPLANATION STABILITY
        # ----------------------------------------------------
        gr.HTML(
            """
            <div class="section-heading">
                Cross-Model SHAP Agreement
            </div>
            <div class="section-description">
                Spearman correlation evaluates the
                complete sensor ranking, whereas the
                Jaccard index evaluates overlap among
                the five most influential sensors.
            </div>
            """
        )
        with gr.Row():
            with gr.Column():
                gr.Plot(
                    value=(
                        reliability_spearman_heatmap
                    ),
                    label=(
                        "Four-Model Spearman Matrix"
                    )
                )
            with gr.Column():
                gr.Plot(
                    value=(
                        reliability_jaccard_heatmap
                    ),
                    label=(
                        "Four-Model Jaccard Matrix"
                    )
                )
        # ----------------------------------------------------
        # RANDOM-SPLIT STABILITY
        # ----------------------------------------------------
        gr.HTML(
            """
            <div class="section-heading">
                Stability Under Different Engine-Level Splits
            </div>
            <div class="section-description">
                Each model was evaluated across multiple
                80-engine / 20-engine train-test partitions.
                Engines were kept entirely within either
                training or testing data to prevent engine
                leakage.
            </div>
            """
        )
        with gr.Row():
            with gr.Column(scale=2):
                gr.Dataframe(
                    value=(
                        reliability_split_display
                    ),
                    headers=[
                        "Model",
                        "Spearman ρ (Mean ± SD)",
                        "Top-5 Jaccard (Mean ± SD)"
                    ],
                    datatype=[
                        "str",
                        "str",
                        "str"
                    ],
                    interactive=False,
                    wrap=True,
                    label=(
                        "Random-Split SHAP Stability"
                    )
                )
            with gr.Column(scale=3):
                gr.Plot(
                    value=(
                        reliability_split_chart
                    ),
                    label=(
                        "Random-Split Stability Comparison"
                    )
                )
        # ----------------------------------------------------
        # INTERPRETATION
        # ----------------------------------------------------
        gr.HTML(
            reliability_interpretation_html
        )
        # ----------------------------------------------------
        # METHODOLOGY NOTE
        # ----------------------------------------------------
        gr.HTML(
            """
            <div class="panel-card">
                <div class="panel-title">
                    Methodological Note
                </div>
                <div style="
                    color:#526875;
                    font-size:12px;
                    line-height:1.7;
                ">
                    Linear Regression uses a linear
                    SHAP-compatible explanation method,
                    while Random Forest and XGBoost use
                    tree-specific SHAP methods.
                    <br><br>
                    SVR is explained using
                    <b>KernelSHAP</b>, a model-agnostic
                    SHAP method suitable for nonlinear
                    black-box models.
                    Representative background and
                    evaluation samples are used for SVR
                    because KernelSHAP is substantially
                    more computationally expensive than
                    TreeSHAP.
                    <br><br>
                    The reliability analysis compares
                    the resulting global feature
                    rankings using the same statistical
                    measures: Spearman rank correlation
                    and Top-5 Jaccard similarity.
                </div>
            </div>
            """
        )
    return page
# ============================================================
# 11. CONFIRMATION
# ============================================================
print("=" * 75)
print("ERMDSF RELIABILITY PAGE — FOUR-MODEL VERSION")
print("=" * 75)
print("✅ Linear Regression included.")
print("✅ Random Forest included.")
print("✅ XGBoost included.")
print("✅ SVR included.")
print(
    "✅ Four-model cross-model Spearman: "
    f"{FOUR_MODEL_CROSS_SPEARMAN_MEAN:.4f} "
    f"± {FOUR_MODEL_CROSS_SPEARMAN_SD:.4f}"
)
print(
    "✅ Four-model Top-5 Jaccard: "
    f"{FOUR_MODEL_CROSS_JACCARD_MEAN:.4f} "
    f"± {FOUR_MODEL_CROSS_JACCARD_SD:.4f}"
)
print(
    "✅ SVR random-split Spearman: "
    "0.9565 ± 0.0187"
)
print(
    "✅ SVR random-split Jaccard: "
    "0.8667 ± 0.1721"
)
print("✅ Reliability page builder updated.")
print("=" * 75)

ERMDSF RELIABILITY PAGE — FOUR-MODEL VERSION
✅ Linear Regression included.
✅ Random Forest included.
✅ XGBoost included.
✅ SVR included.
✅ Four-model cross-model Spearman: 0.9758 ± 0.0146
✅ Four-model Top-5 Jaccard: 0.8333 ± 0.1826
✅ SVR random-split Spearman: 0.9565 ± 0.0187
✅ SVR random-split Jaccard: 0.8667 ± 0.1721
✅ Reliability page builder updated.


In [13]:
# ============================================================
# CELL 10 — CASE STUDIES, FRAMEWORK, METHODOLOGY AND ABOUT
# ============================================================
# ------------------------------------------------------------
# PREPARE OPERATIONAL CASE-STUDY TABLE
# ------------------------------------------------------------
case_study_display = case_dashboard[
    [
        "case_type",
        "engine_id",
        "cycle",
        "actual_RUL",
        "predicted_RUL",
        "lower_95",
        "upper_95",
        "failure_probability",
        "risk_level",
        "top_SHAP_drivers",
        "RUL_reducing_drivers",
        "recommendation"
    ]
].copy()
case_study_display.columns = [
    "Case Type",
    "Engine",
    "Cycle",
    "Actual RUL",
    "Predicted RUL",
    "Lower Interval",
    "Upper Interval",
    "Failure Probability",
    "Risk",
    "Top SHAP Drivers",
    "RUL-Reducing Drivers",
    "Recommended Action"
]
for column in [
    "Actual RUL",
    "Predicted RUL",
    "Lower Interval",
    "Upper Interval"
]:
    case_study_display[column] = (
        case_study_display[column]
        .round(1)
    )
case_study_display[
    "Failure Probability"
] = (
    case_study_display[
        "Failure Probability"
    ]
    .map(
        lambda value:
        f"{value:.1%}"
    )
)
# ------------------------------------------------------------
# CASE-STUDY COLOURS
# ------------------------------------------------------------
CASE_COLORS = {
    "Healthy": COLORS["success"],
    "Warning-stage": COLORS["warning"],
    "Critical": COLORS["critical"]
}
CASE_BACKGROUNDS = {
    "Healthy": "#EDF8EE",
    "Warning-stage": "#FFF8E7",
    "Critical": "#FFF0F0"
}
# ------------------------------------------------------------
# OPERATIONAL CASE-STUDY CARDS
# ------------------------------------------------------------
def create_case_study_cards_html():
    """
    Create one professional card for each selected
    operational case study.
    """
    cards = []
    for _, case_row in case_dashboard.iterrows():
        case_type = str(
            case_row["case_type"]
        )
        case_color = CASE_COLORS.get(
            case_type,
            COLORS["blue"]
        )
        case_background = (
            CASE_BACKGROUNDS.get(
                case_type,
                "#EDF6FC"
            )
        )
        interval_width = (
            float(
                case_row["upper_95"]
            )
            -
            float(
                case_row["lower_95"]
            )
        )
        prediction_error = (
            float(
                case_row["predicted_RUL"]
            )
            -
            float(
                case_row["actual_RUL"]
            )
        )
        if prediction_error > 5:
            prediction_assessment = (
                "The prediction overestimates "
                f"actual RUL by "
                f"{abs(prediction_error):.1f} cycles."
            )
        elif prediction_error < -5:
            prediction_assessment = (
                "The prediction is conservative "
                f"and underestimates actual RUL by "
                f"{abs(prediction_error):.1f} cycles."
            )
        else:
            prediction_assessment = (
                "The predicted and actual RUL values "
                "are closely aligned."
            )
        cards.append(
            f"""
            <div style="
                background:white;
                border:1px solid #DCE5EC;
                border-top:5px solid {case_color};
                border-radius:15px;
                padding:17px;
                box-shadow:
                    0 5px 15px
                    rgba(15,61,89,0.06);
            ">
                <div style="
                    display:flex;
                    justify-content:space-between;
                    align-items:center;
                    gap:10px;
                    margin-bottom:13px;
                ">
                    <div>
                        <div style="
                            color:{case_color};
                            font-size:10px;
                            font-weight:800;
                            letter-spacing:1px;
                        ">
                            {case_type.upper()} CASE
                        </div>
                        <div style="
                            color:{COLORS['text']};
                            font-size:20px;
                            font-weight:800;
                            margin-top:3px;
                        ">
                            Engine
                            {int(case_row['engine_id'])}
                        </div>
                    </div>
                    <div style="
                        background:{case_background};
                        color:{case_color};
                        border-radius:30px;
                        padding:7px 12px;
                        font-size:11px;
                        font-weight:800;
                    ">
                        {case_row['risk_level']}
                    </div>
                </div>
                <div style="
                    display:grid;
                    grid-template-columns:
                        repeat(2, minmax(0, 1fr));
                    gap:9px;
                    margin-bottom:12px;
                ">
                    <div style="
                        background:#F7F9FB;
                        border-radius:10px;
                        padding:10px;
                    ">
                        <div style="
                            color:#7A8995;
                            font-size:9px;
                            font-weight:800;
                        ">
                            SNAPSHOT CYCLE
                        </div>
                        <div style="
                            color:{COLORS['text']};
                            font-size:17px;
                            font-weight:800;
                            margin-top:3px;
                        ">
                            {int(case_row['cycle'])}
                        </div>
                    </div>
                    <div style="
                        background:#F7F9FB;
                        border-radius:10px;
                        padding:10px;
                    ">
                        <div style="
                            color:#7A8995;
                            font-size:9px;
                            font-weight:800;
                        ">
                            PREDICTED RUL
                        </div>
                        <div style="
                            color:{COLORS['text']};
                            font-size:17px;
                            font-weight:800;
                            margin-top:3px;
                        ">
                            {case_row['predicted_RUL']:.1f}
                        </div>
                    </div>
                    <div style="
                        background:#F7F9FB;
                        border-radius:10px;
                        padding:10px;
                    ">
                        <div style="
                            color:#7A8995;
                            font-size:9px;
                            font-weight:800;
                        ">
                            ACTUAL RUL
                        </div>
                        <div style="
                            color:{COLORS['text']};
                            font-size:17px;
                            font-weight:800;
                            margin-top:3px;
                        ">
                            {case_row['actual_RUL']:.1f}
                        </div>
                    </div>
                    <div style="
                        background:#F7F9FB;
                        border-radius:10px;
                        padding:10px;
                    ">
                        <div style="
                            color:#7A8995;
                            font-size:9px;
                            font-weight:800;
                        ">
                            FAILURE PROBABILITY
                        </div>
                        <div style="
                            color:{case_color};
                            font-size:17px;
                            font-weight:800;
                            margin-top:3px;
                        ">
                            {case_row['failure_probability']:.1%}
                        </div>
                    </div>
                </div>
                <div style="
                    background:{case_background};
                    border-radius:10px;
                    padding:11px;
                    margin-bottom:10px;
                    color:#425865;
                    font-size:11px;
                    line-height:1.55;
                ">
                    <b>
                        Research interval:
                    </b>
                    {case_row['lower_95']:.1f}
                    –
                    {case_row['upper_95']:.1f}
                    cycles
                    <br>
                    Interval width:
                    {interval_width:.1f}
                    cycles
                </div>
                <div style="
                    color:#526875;
                    font-size:11px;
                    line-height:1.65;
                ">
                    <b>Top SHAP drivers:</b>
                    {case_row['top_SHAP_drivers']}
                    <br><br>
                    <b>RUL-reducing drivers:</b>
                    {case_row['RUL_reducing_drivers']}
                    <br><br>
                    <b>Prediction assessment:</b>
                    {prediction_assessment}
                </div>
                <div style="
                    background:#F2F8FC;
                    border-left:4px solid {case_color};
                    border-radius:9px;
                    padding:11px;
                    margin-top:13px;
                    color:#34566E;
                    font-size:11px;
                    line-height:1.55;
                ">
                    <b>Recommended action:</b>
                    {case_row['recommendation']}
                </div>
            </div>
            """
        )
    return (
        """
        <div style="
            display:grid;
            grid-template-columns:
                repeat(3, minmax(0, 1fr));
            gap:14px;
        ">
        """
        +
        "".join(cards)
        +
        """
        </div>
        """
    )
# ------------------------------------------------------------
# CASE-STUDY COMPARISON CHART
# ------------------------------------------------------------
def create_case_study_comparison_chart():
    """
    Compare actual RUL, predicted RUL and uncertainty
    for the three operational cases.
    """
    chart_data = case_dashboard.copy()
    chart_data[
        "Case Label"
    ] = (
        chart_data["case_type"]
        +
        "<br>Engine "
        +
        chart_data["engine_id"]
        .astype(int)
        .astype(str)
    )
    lower_error = (
        chart_data["predicted_RUL"]
        -
        chart_data["lower_95"]
    ).clip(lower=0)
    upper_error = (
        chart_data["upper_95"]
        -
        chart_data["predicted_RUL"]
    ).clip(lower=0)
    figure = go.Figure()
    figure.add_trace(
        go.Bar(
            name="Actual RUL",
            x=chart_data[
                "Case Label"
            ],
            y=chart_data[
                "actual_RUL"
            ],
            marker_color="#A9BBC8",
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Actual RUL: %{y:.1f} cycles"
                "<extra></extra>"
            )
        )
    )
    figure.add_trace(
        go.Bar(
            name="Predicted RUL",
            x=chart_data[
                "Case Label"
            ],
            y=chart_data[
                "predicted_RUL"
            ],
            marker_color=[
                CASE_COLORS.get(
                    case_type,
                    COLORS["blue"]
                )
                for case_type
                in chart_data["case_type"]
            ],
            error_y=dict(
                type="data",
                symmetric=False,
                array=upper_error,
                arrayminus=lower_error,
                color="#6E7E8B",
                thickness=1.5,
                width=5
            ),
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Predicted RUL: %{y:.1f} cycles"
                "<extra></extra>"
            )
        )
    )
    figure.add_hline(
        y=maintenance_threshold,
        line_dash="dash",
        line_color=COLORS["high"],
        line_width=2,
        annotation_text=(
            f"{maintenance_threshold}-cycle "
            "maintenance threshold"
        ),
        annotation_position="top right"
    )
    figure.update_layout(
        title=dict(
            text=(
                "Operational Case-Study "
                "RUL Comparison"
            ),
            x=0.04,
            font=dict(
                size=18,
                color=COLORS["text"]
            )
        ),
        xaxis_title=None,
        yaxis_title="RUL (cycles)",
        barmode="group",
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=60,
            r=25,
            t=70,
            b=70
        ),
        height=480,
        legend=dict(
            orientation="h",
            y=-0.18,
            x=0.5,
            xanchor="center"
        )
    )
    figure.update_yaxes(
        gridcolor="#E7EDF2",
        zeroline=False
    )
    return figure
# ------------------------------------------------------------
# CASE-STUDY INTERPRETATION
# ------------------------------------------------------------
def create_case_study_interpretation_html():
    """
    Explain the operational value of the selected cases.
    """
    healthy_row = case_dashboard[
        case_dashboard["case_type"]
        ==
        "Healthy"
    ].iloc[0]
    warning_row = case_dashboard[
        case_dashboard["case_type"]
        ==
        "Warning-stage"
    ].iloc[0]
    critical_row = case_dashboard[
        case_dashboard["case_type"]
        ==
        "Critical"
    ].iloc[0]
    return f"""
    <div class="panel-card">
        <div class="panel-title">
            Operational Case-Study Interpretation
        </div>
        <div style="
            color:#526875;
            font-size:12px;
            line-height:1.75;
        ">
            The
            <b>Healthy</b>
            case demonstrates that a high point prediction,
            low failure probability and positive sensor
            contributions support continued operation with
            routine condition monitoring.
            <br><br>
            The
            <b>Warning-stage</b>
            case shows how a moderate point prediction can
            still lead to a High operational-risk category
            when the lower uncertainty bound approaches the
            failure region and multiple sensors strongly
            reduce predicted RUL.
            <br><br>
            The
            <b>Critical</b>
            case demonstrates the importance of combining
            point prediction, failure probability, uncertainty
            and local SHAP evidence. Engine
            <b>
                {int(critical_row['engine_id'])}
            </b>
            has a predicted RUL of
            <b>
                {critical_row['predicted_RUL']:.1f}
                cycles
            </b>,
            a failure probability of
            <b>
                {critical_row['failure_probability']:.1%}
            </b>
            and dominant RUL-reducing contributions from
            <b>
                {critical_row['RUL_reducing_drivers']}
            </b>.
            <br><br>
            Together, the three cases illustrate how the
            framework translates technical model outputs into
            transparent maintenance recommendations for
            different operational conditions.
        </div>
    </div>
    """
# ------------------------------------------------------------
# ERMDSF FRAMEWORK HTML
# ------------------------------------------------------------
def create_framework_diagram_html():
    """
    Create the complete ERMDSF framework as a visual flow.
    """
    return f"""
    <div style="
        background:white;
        border:1px solid #DCE5EC;
        border-radius:16px;
        padding:22px;
        box-shadow:
            0 5px 15px
            rgba(15,61,89,0.06);
    ">
        <div style="
            display:grid;
            grid-template-columns:
                repeat(7, minmax(0, 1fr));
            align-items:stretch;
            gap:8px;
        ">
            <div style="
                background:#EDF6FC;
                border-top:5px solid {COLORS['blue']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ①
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    DATA ACQUISITION
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    NASA C-MAPSS FD001 sensor measurements,
                    operating settings and engine cycles
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#EDF6FC;
                border-top:5px solid {COLORS['blue']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ②
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    RUL PREDICTION
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Linear Regression, Random Forest and
                    XGBoost regression models
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#F2F8FC;
                border-top:5px solid {COLORS['blue']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ③
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    EXPLAINABILITY
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Global and local SHAP explanations,
                    sensor ranking and prediction drivers
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#EDF8EE;
                border-top:5px solid {COLORS['success']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ④
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    RELIABILITY TESTING
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Spearman, Jaccard, split-state,
                    hyperparameter and noise robustness
                </div>
            </div>
        </div>
        <div style="
            display:flex;
            justify-content:center;
            padding:13px 0;
            color:#87A0B1;
            font-size:25px;
        ">
            ↓
        </div>
        <div style="
            display:grid;
            grid-template-columns:
                repeat(7, minmax(0, 1fr));
            align-items:stretch;
            gap:8px;
        ">
            <div style="
                background:#FFF8E7;
                border-top:5px solid {COLORS['warning']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ⑤
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    UNCERTAINTY
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Split conformal prediction intervals
                    and calibrated coverage assessment
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#FFF5E9;
                border-top:5px solid {COLORS['high']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ⑥
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    RISK ASSESSMENT
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Transparent Low, Medium, High and
                    Critical operational categories
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#FFF0F0;
                border-top:5px solid {COLORS['critical']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ⑦
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    MAINTENANCE DECISION
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Continue operation, monitor,
                    schedule inspection or engineering review
                </div>
            </div>
            <div style="
                display:flex;
                align-items:center;
                justify-content:center;
                color:#87A0B1;
                font-size:24px;
            ">
                →
            </div>
            <div style="
                background:#EDF6FC;
                border-top:5px solid {COLORS['blue']};
                border-radius:12px;
                padding:14px 10px;
                text-align:center;
            ">
                <div style="
                    font-size:24px;
                    margin-bottom:8px;
                ">
                    ⑧
                </div>
                <div style="
                    color:{COLORS['text']};
                    font-size:12px;
                    font-weight:800;
                ">
                    FLEET PRIORITIZATION
                </div>
                <div style="
                    color:#617383;
                    font-size:10px;
                    line-height:1.5;
                    margin-top:7px;
                ">
                    Ordered inspection queue under
                    limited maintenance capacity
                </div>
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# FRAMEWORK COMPONENT TABLE
# ------------------------------------------------------------
framework_component_table = pd.DataFrame({
    "Framework Component": [
        "Point prediction",
        "Prediction uncertainty",
        "Explainability",
        "Explanation reliability",
        "Operational interpretation",
        "Fleet planning"
    ],
    "Method": [
        "XGBoost Regressor",
        "95% split conformal prediction interval",
        "Local and global SHAP",
        (
            "Spearman, Jaccard, split-state, "
            "hyperparameter and noise tests"
        ),
        "Transparent operational risk matrix",
        "Risk-based maintenance-priority ranking"
    ],
    "Operational Output": [
        "Estimated RUL in cycles",
        "Lower and upper plausible RUL bounds",
        (
            "Sensors increasing or reducing "
            "predicted RUL"
        ),
        (
            "Evidence that explanations remain "
            "stable"
        ),
        "Low, Medium, High or Critical risk",
        (
            "Ordered engines requiring "
            "maintenance attention"
        )
    ]
})
# ------------------------------------------------------------
# FRAMEWORK CONTRIBUTION HTML
# ------------------------------------------------------------
def create_framework_contribution_html():
    """
    Explain the thesis contribution represented by ERMDSF.
    """
    return """
    <div class="panel-card">
        <div class="panel-title">
            Main Research Contribution
        </div>
        <div style="
            color:#526875;
            font-size:12px;
            line-height:1.75;
        ">
            The ERMDSF framework extends conventional
            Remaining Useful Life prediction by integrating
            five connected forms of evidence:
            <br><br>
            <b>1. Predictive evidence</b> —
            estimated RUL and failure probability.
            <br>
            <b>2. Uncertainty evidence</b> —
            calibrated conformal prediction intervals.
            <br>
            <b>3. Explainability evidence</b> —
            global and local SHAP sensor contributions.
            <br>
            <b>4. Reliability evidence</b> —
            explanation stability across models, data splits,
            hyperparameters and sensor perturbations.
            <br>
            <b>5. Operational evidence</b> —
            transparent risk classification, maintenance
            recommendations and fleet-priority ranking.
            <br><br>
            The novelty is therefore not a single predictive
            algorithm. It is the integration of prediction,
            uncertainty, explanation reliability and
            operational decision support into one transparent
            maintenance framework.
        </div>
    </div>
    """
# ------------------------------------------------------------
# METHODOLOGY HTML
# ------------------------------------------------------------
def create_methodology_html():
    """
    Summarize the methodology used in the thesis.
    """
    return f"""
    <div style="
        display:grid;
        grid-template-columns:
            repeat(2, minmax(0, 1fr));
        gap:14px;
    ">
        <div class="panel-card">
            <div class="panel-title">
                Dataset and Preprocessing
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.7;
            ">
                <b>Dataset:</b>
                NASA C-MAPSS FD001 turbofan-engine
                degradation dataset.
                <br><br>
                <b>Training observations:</b>
                {len(train_dashboard):,}
                <br>
                <b>Engine-wise splitting:</b>
                used to prevent information leakage
                between training and test engines.
                <br>
                <b>RUL target:</b>
                Remaining Useful Life capped at
                125 cycles.
                <br>
                <b>Failure-classification threshold:</b>
                RUL below
                {maintenance_threshold}
                cycles.
            </div>
        </div>
        <div class="panel-card">
            <div class="panel-title">
                Predictive Models
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.7;
            ">
                <b>Baseline model:</b>
                Linear Regression.
                <br><br>
                <b>Ensemble models:</b>
                Random Forest Regressor and
                XGBoost Regressor.
                <br><br>
                <b>Primary regression metrics:</b>
                RMSE and MAE.
                <br><br>
                <b>Classification metrics:</b>
                Accuracy and F1-score for detecting
                engines with RUL below
                {maintenance_threshold}
                cycles.
            </div>
        </div>
        <div class="panel-card">
            <div class="panel-title">
                Explainability and Reliability
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.7;
            ">
                <b>Global SHAP:</b>
                identifies sensors with the strongest
                fleet-level influence.
                <br><br>
                <b>Local SHAP:</b>
                explains individual engine predictions.
                <br><br>
                <b>Ranking reliability:</b>
                evaluated using Spearman rank correlation
                and Top-k Jaccard similarity.
                <br><br>
                <b>Robustness testing:</b>
                split states, model hyperparameters and
                simulated Gaussian sensor noise.
            </div>
        </div>
        <div class="panel-card">
            <div class="panel-title">
                Operational Decision Support
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.7;
            ">
                <b>Prediction uncertainty:</b>
                95% split conformal intervals.
                <br><br>
                <b>Operational categories:</b>
                Low, Medium, High and Critical.
                <br><br>
                <b>Maintenance outputs:</b>
                continue operation, routine monitoring,
                priority inspection planning or immediate
                engineering review.
                <br><br>
                <b>Fleet planning:</b>
                engines ranked under configurable
                inspection capacity.
            </div>
        </div>
    </div>
    """
# ------------------------------------------------------------
# ABOUT PROJECT HTML
# ------------------------------------------------------------
def create_about_project_html():
    """
    Create project and thesis information panel.
    """
    return """
    <div style="
        display:grid;
        grid-template-columns:
            1.2fr 1fr;
        gap:14px;
    ">
        <div class="panel-card">
            <div class="panel-title">
                Thesis Project
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.75;
            ">
                <b>Title</b>
                <br>
                Remaining Useful Life Prediction of
                Aircraft Turbofan Engines Using
                Machine Learning and SHAP
                <br><br>
                <b>Author</b>
                <br>
                Muhammad Dilawar
                <br><br>
                <b>Supervisor</b>
                <br>
                Prof. Dr. Maik Weber
                <br><br>
                <b>Institution</b>
                <br>
                Hochschule Trier —
                Trier University of Applied Sciences
                <br><br>
                <b>Dataset</b>
                <br>
                NASA C-MAPSS FD001
            </div>
        </div>
        <div class="panel-card">
            <div class="panel-title">
                Dashboard Scope
            </div>
            <div style="
                color:#526875;
                font-size:12px;
                line-height:1.75;
            ">
                This platform is an academic research
                prototype that demonstrates how machine
                learning outputs can be translated into
                transparent aircraft-engine maintenance
                decision support.
                <br><br>
                It includes:
                <br>
                • Fleet health monitoring
                <br>
                • Individual engine analysis
                <br>
                • SHAP-based prediction explanation
                <br>
                • Explanation-reliability evidence
                <br>
                • Conformal prediction uncertainty
                <br>
                • Capacity-based maintenance planning
                <br>
                • Operational case studies
            </div>
        </div>
    </div>
    <div class="danger-box" style="margin-top:15px;">
        <b>Important limitation:</b>
        ERMDSF is an academic research prototype.
        It is not approved for aircraft maintenance release,
        airworthiness decisions, operational dispatch,
        continuing-airworthiness management or replacement
        of certified engineering procedures.
    </div>
    """
# ------------------------------------------------------------
# RESEARCH LIMITATIONS HTML
# ------------------------------------------------------------
def create_research_limitations_html():
    """
    Summarize important scientific and operational limitations.
    """
    return """
    <div class="panel-card">
        <div class="panel-title">
            Research Limitations
        </div>
        <div style="
            color:#526875;
            font-size:12px;
            line-height:1.75;
        ">
            <b>Simulated dataset:</b>
            NASA C-MAPSS represents simulated degradation
            rather than real airline maintenance records.
            <br><br>
            <b>Generic sensor labels:</b>
            FD001 uses anonymized sensor names, which limits
            direct physical interpretation without additional
            literature-based mapping.
            <br><br>
            <b>Wide uncertainty intervals:</b>
            The conformal intervals achieved strong empirical
            coverage but remained operationally broad.
            <br><br>
            <b>Snapshot-based dashboard:</b>
            Fleet prioritization is based on selected engine
            snapshots rather than a continuously streaming
            health-monitoring system.
            <br><br>
            <b>Decision-support role:</b>
            Risk categories and maintenance recommendations
            support interpretation but do not constitute
            certified maintenance instructions.
        </div>
    </div>
    """
# ------------------------------------------------------------
# BUILD CELL 10 OBJECTS
# ------------------------------------------------------------
case_study_cards_html = (
    create_case_study_cards_html()
)
case_study_comparison_chart = (
    create_case_study_comparison_chart()
)
case_study_interpretation_html = (
    create_case_study_interpretation_html()
)
framework_diagram_html = (
    create_framework_diagram_html()
)
framework_contribution_html = (
    create_framework_contribution_html()
)
methodology_html = (
    create_methodology_html()
)
about_project_html = (
    create_about_project_html()
)
research_limitations_html = (
    create_research_limitations_html()
)
# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------
print("=" * 65)
print("ERMDSF FRAMEWORK AND CASE-STUDY COMPONENTS")
print("=" * 65)
print("✅ Operational case-study table created.")
print("✅ Operational case-study cards created.")
print("✅ Case-study comparison chart created.")
print("✅ Case-study interpretation created.")
print("✅ ERMDSF framework diagram created.")
print("✅ Framework component table created.")
print("✅ Research contribution panel created.")
print("✅ Methodology panel created.")
print("✅ About Project panel created.")
print("✅ Research limitations panel created.")
print("=" * 65)

ERMDSF FRAMEWORK AND CASE-STUDY COMPONENTS
✅ Operational case-study table created.
✅ Operational case-study cards created.
✅ Case-study comparison chart created.
✅ Case-study interpretation created.
✅ ERMDSF framework diagram created.
✅ Framework component table created.
✅ Research contribution panel created.
✅ Methodology panel created.
✅ About Project panel created.
✅ Research limitations panel created.


In [14]:
                            gr.HTML(
                                f"""
                                <div class="panel-card">
                                    <div class="panel-title">
                                        Explanation Reliability
                                    </div>
                                    <div style="
                                        color:#526875;
                                        font-size:12px;
                                        line-height:1.7;
                                    ">
                                        Cross-model Spearman:
                                        <b>
                                            {cross_model_spearman:.3f}
                                        </b>
                                        <br>
                                        Cross-model Top-5 Jaccard:
                                        <b>
                                            {cross_model_jaccard:.3f}
                                        </b>
                                        <br>
                                        Sensor-noise Spearman:
                                        <b>
                                            {noise_spearman:.3f}
                                        </b>
                                        <br>
                                        Sensor-noise Top-5 Jaccard:
                                        <b>
                                            {noise_jaccard:.3f}
                                        </b>
                                        <br><br>
                                        Global sensor rankings remained
                                        highly stable across models,
                                        engine splits, hyperparameters
                                        and simulated sensor noise.
                                    </div>
                                </div>
                                """
                            )

In [15]:
# ============================================================
# CELL 11 — COMPLETE ERMDSF APPLICATION INTERFACE
# ORIGINAL HEADER + 8 WORKING SIDEBAR PAGES
# ============================================================
import gradio as gr
# ------------------------------------------------------------
# CLOSE PREVIOUS DEMO OBJECT IF IT EXISTS
# ------------------------------------------------------------
try:
    demo.close()
except Exception:
    pass
# ============================================================
# PREPARE ENGINE SELECTORS
# ============================================================
engine_choices = sorted(
    [int(engine) for engine in engine_cache.keys()]
)
if len(engine_choices) == 0:
    raise RuntimeError(
        "No engines were found in engine_cache."
    )
initial_engine = engine_choices[0]
# ============================================================
# INITIAL ENGINE ANALYSIS OUTPUTS
# ============================================================
(
    initial_engine_summary,
    initial_engine_gauge,
    initial_engine_recommendation,
    initial_engine_shap_table,
    initial_engine_shap_bar,
    initial_engine_waterfall,
    initial_engine_interpretation
) = update_engine_analysis(
    initial_engine
)
# ============================================================
# INITIAL EXPLAINABILITY OUTPUTS
# ============================================================
(
    initial_explain_table,
    initial_explain_bar,
    initial_explain_waterfall,
    initial_explain_interpretation
) = update_explainability_engine(
    initial_engine
)
# ============================================================
# NAVIGATION FUNCTION
# ============================================================
def show_page(page_number):
    """
    Show one ERMDSF page and hide all others.
    Eight pages are available.
    """
    return tuple(
        gr.update(
            visible=(index == page_number)
        )
        for index in range(8)
    )
# ============================================================
# BUILD APPLICATION
# ============================================================
with gr.Blocks(
    title="ERMDSF — Fleet Health Monitoring"
) as demo:
    # ========================================================
    # ORIGINAL ERMDSF HEADER
    # ========================================================
    gr.HTML(
        """
        <div class="app-header">
            <div class="brand-row">
                <div class="brand-mark">
                    ✈
                </div>
                <div>
                    <div class="brand-title">
                        ERMDSF
                    </div>
                    <div class="brand-subtitle">
                        Explainable Remaining Useful Life &
                        Maintenance Decision-Support Framework
                    </div>
                </div>
                <div class="header-status">
                    RESEARCH PROTOTYPE
                </div>
            </div>
        </div>
        """
    )
    # ========================================================
    # SIDEBAR + MAIN CONTENT
    # ========================================================
    with gr.Row():
        # ====================================================
        # SIDEBAR
        # ====================================================
        with gr.Column(
            scale=1,
            min_width=260,
            elem_classes=["sidebar-shell"]
        ):
            gr.HTML(
                """
                <div class="sidebar-brand">
                    <div class="sidebar-brand-title">
                        Control Center
                    </div>
                    <div class="sidebar-brand-subtitle">
                        FLEET HEALTH MONITORING
                    </div>
                </div>
                """
            )
            nav_dashboard = gr.Button(
                "⌂ Dashboard",
                elem_classes=["nav-btn"]
            )
            nav_fleet = gr.Button(
                "▦ Fleet Health",
                elem_classes=["nav-btn"]
            )
            nav_engine = gr.Button(
                "◉ Engine Analysis",
                elem_classes=["nav-btn"]
            )
            nav_maintenance = gr.Button(
                "⚒ Maintenance Planner",
                elem_classes=["nav-btn"]
            )
            nav_explainability = gr.Button(
                "≋ Explainability",
                elem_classes=["nav-btn"]
            )
            nav_reliability = gr.Button(
                "◇ Reliability Center",
                elem_classes=["nav-btn"]
            )
            nav_cases = gr.Button(
                "▤ Operational Cases",
                elem_classes=["nav-btn"]
            )
            # NEW ABOUT BUTTON
            nav_about = gr.Button(
                "ⓘ About",
                elem_classes=["nav-btn"]
            )
            gr.HTML(
                """
                <div style="
                    margin-top:20px;
                    padding:14px 10px;
                    border-top:
                        1px solid rgba(255,255,255,0.12);
                    color:#AFCBDD;
                    font-size:10px;
                    line-height:1.65;
                ">
                    NASA C-MAPSS FD001
                    <br><br>
                    Prediction · Uncertainty ·
                    Explainability · Reliability ·
                    Decision Support
                </div>
                """
            )
        # ====================================================
        # MAIN CONTENT AREA
        # ====================================================
        with gr.Column(
            scale=5,
            elem_classes=["main-shell"]
        ):
            # =================================================
            # PAGE 1 — DASHBOARD
            # =================================================
            with gr.Column(
                visible=True
            ) as page_dashboard:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        FLEET OVERVIEW
                    </div>
                    <div class="page-title">
                        Dashboard
                    </div>
                    <div class="page-description">
                        Fleet-level overview of predicted
                        Remaining Useful Life, operational risk,
                        alerts and recommended actions.
                    </div>
                    """
                )
                gr.HTML(
                    home_kpi_html
                )
                with gr.Row():
                    gr.Plot(
                        value=fleet_risk_donut,
                        label="Fleet Risk Distribution"
                    )
                    gr.Plot(
                        value=fleet_rul_histogram,
                        label="Predicted RUL Distribution"
                    )
                with gr.Row():
                    with gr.Column():
                        gr.HTML(
                            home_alerts_html
                        )
                    with gr.Column():
                        gr.HTML(
                            home_recommendations_html
                        )
            # =================================================
            # PAGE 2 — FLEET HEALTH
            # =================================================
            with gr.Column(
                visible=False
            ) as page_fleet:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        FLEET MONITORING
                    </div>
                    <div class="page-title">
                        Fleet Health
                    </div>
                    <div class="page-description">
                        Comparative health, Remaining Useful
                        Life, uncertainty and failure-probability
                        assessment across monitored engines.
                    </div>
                    """
                )
                gr.HTML(
                    fleet_health_kpi_html
                )
                with gr.Row():
                    gr.Plot(
                        value=fleet_rul_ranking_chart,
                        label="Fleet RUL Ranking"
                    )
                    gr.Plot(
                        value=actual_vs_predicted_chart,
                        label="Actual vs Predicted RUL"
                    )
                with gr.Row():
                    gr.Plot(
                        value=failure_probability_chart,
                        label="Failure Probability"
                    )
                    gr.Plot(
                        value=uncertainty_interval_chart,
                        label="Prediction Uncertainty"
                    )
                gr.Dataframe(
                    value=fleet_health_table,
                    interactive=False,
                    wrap=True,
                    label="Fleet Health Table"
                )
                gr.HTML(
                    fleet_health_interpretation_html
                )
            # =================================================
            # PAGE 3 — ENGINE ANALYSIS
            # =================================================
            with gr.Column(
                visible=False
            ) as page_engine:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        INDIVIDUAL ENGINE
                    </div>
                    <div class="page-title">
                        Engine Analysis
                    </div>
                    <div class="page-description">
                        Engine-specific RUL prediction,
                        uncertainty, operational risk,
                        SHAP drivers and maintenance
                        interpretation.
                    </div>
                    """
                )
                with gr.Column(
                    elem_classes=["engine-control-card"]
                ):
                    engine_selector = gr.Dropdown(
                        choices=engine_choices,
                        value=initial_engine,
                        label="Select Engine",
                        interactive=True
                    )
                engine_summary = gr.HTML(
                    initial_engine_summary
                )
                with gr.Row():
                    engine_gauge = gr.Plot(
                        value=initial_engine_gauge,
                        label="RUL Health Gauge"
                    )
                    engine_recommendation = gr.HTML(
                        initial_engine_recommendation
                    )
                engine_shap_table = gr.Dataframe(
                    value=initial_engine_shap_table,
                    interactive=False,
                    wrap=True,
                    label="Top Local SHAP Contributions"
                )
                with gr.Row():
                    engine_shap_bar = gr.Plot(
                        value=initial_engine_shap_bar,
                        label="Local SHAP Contributions"
                    )
                    engine_waterfall = gr.Plot(
                        value=initial_engine_waterfall,
                        label="SHAP Waterfall"
                    )
                engine_interpretation = gr.HTML(
                    initial_engine_interpretation
                )
                engine_selector.change(
                    fn=update_engine_analysis,
                    inputs=[
                        engine_selector
                    ],
                    outputs=[
                        engine_summary,
                        engine_gauge,
                        engine_recommendation,
                        engine_shap_table,
                        engine_shap_bar,
                        engine_waterfall,
                        engine_interpretation
                    ]
                )
            # =================================================
            # PAGE 4 — MAINTENANCE PLANNER
            # =================================================
            with gr.Column(
                visible=False
            ) as page_maintenance:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        DECISION SUPPORT
                    </div>
                    <div class="page-title">
                        Maintenance Planner
                    </div>
                    <div class="page-description">
                        Prioritize engines for inspection
                        according to operational risk and
                        available maintenance capacity.
                    </div>
                    """
                )
                with gr.Row():
                    planner_capacity = gr.Slider(
                        minimum=1,
                        maximum=max(
                            1,
                            len(fleet_dashboard)
                        ),
                        value=min(
                            10,
                            max(
                                1,
                                len(fleet_dashboard)
                            )
                        ),
                        step=1,
                        label="Inspection Capacity"
                    )
                    planner_risks = gr.CheckboxGroup(
                        choices=[
                            "Critical",
                            "High",
                            "Medium",
                            "Low"
                        ],
                        value=[
                            "Critical",
                            "High",
                            "Medium",
                            "Low"
                        ],
                        label="Include Risk Categories"
                    )
                planner_run = gr.Button(
                    "Generate Maintenance Plan",
                    variant="primary"
                )
                planner_summary = gr.HTML(
                    """
                    <div class="panel-card">
                        <div class="panel-title">
                            Maintenance Plan
                        </div>
                        <div style="
                            color:#526875;
                            font-size:12px;
                        ">
                            Choose maintenance capacity and
                            risk categories, then generate
                            the inspection-priority plan.
                        </div>
                    </div>
                    """
                )
                planner_table = gr.Dataframe(
                    value=empty_maintenance_table(),
                    interactive=False,
                    wrap=True,
                    label="Selected Maintenance Priorities"
                )
                planner_chart = gr.Plot(
                    value=empty_maintenance_chart(
                        "Generate a maintenance plan"
                    ),
                    label="Maintenance Priority"
                )
                planner_file = gr.File(
                    label="Exported Maintenance Plan"
                )
                planner_run.click(
                    fn=build_maintenance_plan,
                    inputs=[
                        planner_capacity,
                        planner_risks
                    ],
                    outputs=[
                        planner_summary,
                        planner_table,
                        planner_chart,
                        planner_file
                    ]
                )
            # =================================================
            # PAGE 5 — EXPLAINABILITY
            # =================================================
            with gr.Column(
                visible=False
            ) as page_explainability:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        MODEL EXPLANATION
                    </div>
                    <div class="page-title">
                        Explainability
                    </div>
                    <div class="page-description">
                        Global and local SHAP evidence
                        describing which variables influence
                        Remaining Useful Life predictions.
                    </div>
                    """
                )
                gr.HTML(
                    explainability_kpi_html
                )
                with gr.Row():
                    gr.Plot(
                        value=global_shap_bar_chart,
                        label="Global SHAP Importance"
                    )
                    gr.Plot(
                        value=fleet_shap_heatmap,
                        label="Fleet SHAP Heatmap"
                    )
                gr.HTML(
                    global_explainability_html
                )
                gr.HTML(
                    """
                    <div class="section-heading">
                        Local Engine Explanation
                    </div>
                    """
                )
                explain_engine_selector = gr.Dropdown(
                    choices=engine_choices,
                    value=initial_engine,
                    label="Select Engine",
                    interactive=True
                )
                explain_table = gr.Dataframe(
                    value=initial_explain_table,
                    interactive=False,
                    wrap=True,
                    label="Local SHAP Contributions"
                )
                with gr.Row():
                    explain_bar = gr.Plot(
                        value=initial_explain_bar,
                        label="Local SHAP Bar"
                    )
                    explain_waterfall = gr.Plot(
                        value=initial_explain_waterfall,
                        label="Local SHAP Waterfall"
                    )
                explain_interpretation = gr.HTML(
                    initial_explain_interpretation
                )
                explain_engine_selector.change(
                    fn=update_explainability_engine,
                    inputs=[
                        explain_engine_selector
                    ],
                    outputs=[
                        explain_table,
                        explain_bar,
                        explain_waterfall,
                        explain_interpretation
                    ]
                )
            # =================================================
            # PAGE 6 — RELIABILITY CENTER
            # =================================================
            page_reliability = (
                build_reliability_page()
            )
            # =================================================
            # PAGE 7 — OPERATIONAL CASES
            # =================================================
            with gr.Column(
                visible=False
            ) as page_cases:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        RESEARCH CASE STUDIES
                    </div>
                    <div class="page-title">
                        Operational Cases
                    </div>
                    <div class="page-description">
                        Selected healthy, warning-stage and
                        critical examples illustrating the full
                        ERMDSF decision-support chain.
                    </div>
                    """
                )
                gr.HTML(
                    case_study_cards_html
                )
                gr.HTML(
                    """
                    <div class="section-heading">
                        Case Comparison
                    </div>
                    """
                )
                gr.Plot(
                    value=case_study_comparison_chart,
                    label="Case Study Comparison"
                )
                gr.Dataframe(
                    value=case_study_display,
                    interactive=False,
                    wrap=True,
                    label="Operational Case Study Data"
                )
                gr.HTML(
                    case_study_interpretation_html
                )
                gr.HTML(
                    """
                    <div class="section-heading">
                        ERMDSF Decision-Support Framework
                    </div>
                    """
                )
                gr.HTML(
                    framework_diagram_html
                )
                gr.Dataframe(
                    value=framework_component_table,
                    interactive=False,
                    wrap=True,
                    label="Framework Components"
                )
                gr.HTML(
                    framework_contribution_html
                )
            # =================================================
            # PAGE 8 — ABOUT
            # =================================================
            with gr.Column(
                visible=False
            ) as page_about:
                gr.HTML(
                    """
                    <div class="page-eyebrow">
                        PROJECT INFORMATION
                    </div>
                    <div class="page-title">
                        About ERMDSF
                    </div>
                    <div class="page-description">
                        Research context, methodology,
                        thesis information, system scope
                        and limitations of the ERMDSF
                        research prototype.
                    </div>
                    """
                )
                # ---------------------------------------------
                # THESIS / PROJECT INFORMATION
                # ---------------------------------------------
                gr.HTML(
                    about_project_html
                )
                # ---------------------------------------------
                # METHODOLOGY
                # ---------------------------------------------
                gr.HTML(
                    """
                    <div class="section-heading">
                        Research Methodology
                    </div>
                    <div class="section-description">
                        Summary of the dataset, predictive
                        modelling, explainability,
                        reliability analysis and operational
                        decision-support methodology.
                    </div>
                    """
                )
                gr.HTML(
                    methodology_html
                )
                # ---------------------------------------------
                # LIMITATIONS
                # ---------------------------------------------
                gr.HTML(
                    """
                    <div class="section-heading">
                        Research Limitations
                    </div>
                    """
                )
                gr.HTML(
                    research_limitations_html
                )
    # ========================================================
    # ALL NAVIGABLE PAGES
    # ========================================================
    page_outputs = [
        page_dashboard,       # 0
        page_fleet,           # 1
        page_engine,          # 2
        page_maintenance,     # 3
        page_explainability,  # 4
        page_reliability,     # 5
        page_cases,           # 6
        page_about            # 7
    ]
    # ========================================================
    # SIDEBAR NAVIGATION EVENTS
    # ========================================================
    nav_dashboard.click(
        fn=lambda: show_page(0),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_fleet.click(
        fn=lambda: show_page(1),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_engine.click(
        fn=lambda: show_page(2),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_maintenance.click(
        fn=lambda: show_page(3),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_explainability.click(
        fn=lambda: show_page(4),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_reliability.click(
        fn=lambda: show_page(5),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_cases.click(
        fn=lambda: show_page(6),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
    nav_about.click(
        fn=lambda: show_page(7),
        inputs=None,
        outputs=page_outputs,
        queue=False
    )
# ============================================================
# CONFIRMATION
# ============================================================
print("=" * 72)
print("ERMDSF APPLICATION INTERFACE")
print("=" * 72)
print("✅ Original blue ERMDSF header restored.")
print("✅ Original sidebar design retained.")
print("✅ Dashboard connected.")
print("✅ Fleet Health connected.")
print("✅ Engine Analysis connected.")
print("✅ Maintenance Planner connected.")
print("✅ Explainability connected.")
print("✅ Reliability Center connected.")
print("✅ Operational Cases connected.")
print("✅ About page added and connected.")
print("✅ Dashboard opens by default.")
print("=" * 72)

ERMDSF APPLICATION INTERFACE
✅ Original blue ERMDSF header restored.
✅ Original sidebar design retained.
✅ Dashboard connected.
✅ Fleet Health connected.
✅ Engine Analysis connected.
✅ Maintenance Planner connected.
✅ Explainability connected.
✅ Reliability Center connected.
✅ Operational Cases connected.
✅ About page added and connected.
✅ Dashboard opens by default.


In [16]:
# ============================================================
# CELL 12 — LAUNCH ERMDSF DASHBOARD
# GRADIO 6 COMPATIBLE
# ============================================================
# Enable the Gradio request queue
demo.queue(
    default_concurrency_limit=2,
    max_size=20
)
# Launch the dashboard
launch_result = demo.launch(
    share=True,
    debug=False,
    show_error=True,
    prevent_thread_lock=True,
    css=custom_css,
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="slate"
    )
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://05ee6300986d900ef2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
